In [1]:
# 0. Setup: import libraries and define API key

import requests
import pandas as pd

API_KEY = "c0d18a0f217bfb86aeb4a9b089540c9a77a34f5b"

In [2]:
# 1. Constants (dataset configuration)

STATE_FIPS_CA = "06"
ACS_YEAR = 2024
ACS_BASE = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

# Start with one variable (total population)
VARS = ["NAME", "B01003_001E"]

In [4]:
# 2. Pull all California places (sanity test)

params = {
    "get": ",".join(VARS),                     # Variables to retrieve
    "for": "place:*",                          # All places (cities + CDPs)
    "in": f"state:{STATE_FIPS_CA}",            # California (FIPS 06)
    "key": API_KEY                             # Census API key
}

response = requests.get(ACS_BASE, params=params)
response.raise_for_status()

data = response.json()

ca_places = pd.DataFrame(data[1:], columns=data[0])

# Convert population column to numeric
ca_places["B01003_001E"] = pd.to_numeric(
    ca_places["B01003_001E"], errors="coerce"
)

ca_places.head()

,NAME,B01003_001E,state,place
0,"Acalanes Ridge CDP, California",1006,06,00135
1,"Acampo CDP, California",234,06,00156
2,"Acton CDP, California",7880,06,00212
3,"Adelanto city, California",37964,06,00296
4,"Adin CDP, California",153,06,00310


In [8]:
# 3. Load in unfiltered dataset

zvhi = pd.read_excel("../Data/Ca_Ziit finally loadedp_Home_Value.xlsx", header=1)
zvhi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,1/31/2000,...,4/30/2025,5/31/2025,6/30/2025,7/31/2025,8/31/2025,9/30/2025,10/31/2025,11/30/2025,12/31/2025,1/31/2026
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,204988.3686,...,486136.1090,484000.3527,481473.2672,479455.2354,478365.6216,478562.4520,479293.4729,480341.6326,480930.8672,480543.6336
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,111719.9414,...,529389.7484,532145.3507,534655.0414,536426.4444,537661.4257,540194.5829,545049.8352,550925.6822,557122.8387,561559.0286
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,102728.1881,...,274073.7545,273277.6176,272287.6011,271356.0568,270430.1504,269659.0928,268926.6435,268297.3865,267961.7208,267417.8897
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,172864.4900,...,528122.2293,528512.4396,530570.0400,533309.6749,534970.6966,536204.0719,537227.4679,539139.3570,541823.1780,546584.2974
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,102246.1487,...,270027.6193,269137.4526,268139.3696,267230.2256,266357.8020,265483.6547,264636.7740,263910.1904,263568.2033,263066.0976


In [ ]:
# 4. Filter ZVHI to CA + (San Diego County OR Los Angeles County) and extract unique city names

zvhi_ca = zvhi[zvhi["State"] == "CA"].copy()

zvhi_sd_la = zvhi_ca[zvhi_ca["CountyName"].isin(["San Diego County", "Los Angeles County"])].copy()

zvhi_city_names = sorted(zvhi_sd_la["City"].dropna().unique())

len(zvhi_city_names), zvhi_city_names[:25]

In [ ]:
# 5. Create ACS-style city name format from Zillow city list
    # This makes the ZHVI cities match the ACS city format

zvhi_city_names_acs = [
    f"{city} city, California"
    for city in zvhi_city_names
]

zvhi_city_names_acs[:20]

In [ ]:
# 6. Check how many Zillow cities match ACS places, should be 135
    # 94/135 match

matches = ca_places[ca_places["NAME"].isin(zvhi_city_names_acs)]

len(matches), matches.head()

In [ ]:
# 6a. Single step: report unmatched cities from the simple "City city, California" mapping
    # this shows 30/41 missing cities are CDP (unincorporated)
acs_names = set(ca_places["NAME"].astype(str))
zvhi_raw = sorted(zvhi_city_names)                            # raw city names from Zillow
zvhi_acs = [f"{c} city, California" for c in zvhi_raw]       # current ACS-style list

unmatched = [city for city, acs in zip(zvhi_raw, zvhi_acs) if acs not in acs_names]

print("total_zillow_cities =", len(zvhi_raw))
print("matched_with_simple_city_suffix =", len(zvhi_raw) - len(unmatched))
print("unmatched_count =", len(unmatched))
print("\nFirst 30 unmatched (sample):")
print(unmatched[:30])

In [ ]:
# 6b. Check which unmatched cities exist as CDPs in ACS
    # prints the 30 CDP's
cdp_matches = []

for city in unmatched:
    candidate = f"{city} CDP, California"
    if candidate in acs_names:
        cdp_matches.append(candidate)

len(cdp_matches), cdp_matches[:20]

In [ ]:
# 6c. Remove CDP-resolved cities from unmatched
    # prints the 11 nonCDP's
resolved_cdp_cities = [name.replace(" CDP, California", "") for name in cdp_matches]

still_unmatched = [city for city in unmatched if city not in resolved_cdp_cities]

len(still_unmatched), still_unmatched

In [ ]:
# 6d. Single step: normalized-match the remaining unmatched cities against ACS names
    # normalization
import re

def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(city|cdp|town|village)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

acs_norm_map = {}
for a in acs_names:
    acs_norm_map.setdefault(normalize(a), []).append(a)

still_unmatched = ['Dulzura','La Canada Flintridge','Lakeside','Llano',
                   'Palomar Mountain','Pauma Valley','Pearblossom',
                   'Ranchita','Santa Ysabel','Spring Valley','Warner Springs']

results = {}
for city in still_unmatched:
    n = normalize(city)
    results[city] = acs_norm_map.get(n, [])

results

In [ ]:
# 6e. Final ACS-matched Zillow cities (city + CDP)
final_acs_names = set(matches["NAME"]).union(set(cdp_matches))

len(final_acs_names)

In [ ]:
# 7. Final ACS subset aligned with Zillow cities
    # 124, 4
        # This drops the 11 non census cities to make the list 124 cities

final_acs_df = ca_places[ca_places["NAME"].isin(final_acs_names)].copy()

final_acs_df.shape

In [ ]:
# 8. Create GEOID for consistent merging across years
    # creates clean geographic ID that's reusable across all years

final_acs_df["GEOID"] = final_acs_df["state"] + final_acs_df["place"]

final_acs_df = final_acs_df[["GEOID", "NAME", "B01003_001E"]]

final_acs_df.head()

In [ ]:
# 9. Define ACS 5-year panel years

years = list(range(2009, 2025))
years

In [ ]:
# 9a. Pull ACS total population (B01003_001E) for 2009-2024 into a panel keyed by GEOID
    # Merge w 2009-2024 Population numbers
import time

years = list(range(2009, 2025))
vars_to_get = ["NAME", "B01003_001E"]   # name + total population
panel_df = final_acs_df[["GEOID", "NAME"]].set_index("GEOID").copy()

for yr in years:
    BASE = f"https://api.census.gov/data/{yr}/acs/acs5"
    params = {
        "get": ",".join(vars_to_get),
        "for": "place:*",
        "in": f"state:{STATE_FIPS_CA}",
        "key": API_KEY
    }
    print(f"Fetching {yr}...", end=" ", flush=True)
    r = requests.get(BASE, params=params)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    # create GEOID and keep only places we care about
    df["GEOID"] = df["state"] + df["place"]
    df["B01003_001E"] = pd.to_numeric(df["B01003_001E"], errors="coerce")
    df_sub = df.set_index("GEOID")[["NAME", "B01003_001E"]].reindex(panel_df.index)
    panel_df[f"pop_{yr}"] = df_sub["B01003_001E"]
    print("done")
    time.sleep(1)  # polite pause to avoid hammering the API

# quick checks
panel_df.reset_index(inplace=True)
print("\npanel shape:", panel_df.shape)
panel_df.head()

In [ ]:
# 10. Convert population panel to long format

panel_long = panel_df.melt(
    id_vars=["GEOID", "NAME"],
    var_name="year",
    value_name="population"
)

# Clean year column
panel_long["year"] = panel_long["year"].str.replace("pop_", "").astype(int)

panel_long.head()

In [ ]:
# 11. Create mapping from normalized city name to GEOID
    # confirm the city + unique GEOID works
def normalize_city(s):
    return s.lower().replace(" city, california", "").replace(" cdp, california", "").strip()

acs_lookup = {
    normalize_city(name): geoid
    for geoid, name in zip(panel_df["GEOID"], panel_df["NAME"])
}

list(acs_lookup.items())[:10]

In [ ]:
# 12. Attach ACS GEOID to each Zillow row using Zillow City
    # Attach unique GEOID to each Zillow city

zvhi_sd_la["city_norm"] = zvhi_sd_la["City"].astype(str).str.lower().str.strip()
zvhi_sd_la["GEOID"] = zvhi_sd_la["city_norm"].map(acs_lookup)

# Check match rate
total = len(zvhi_sd_la)
matched = zvhi_sd_la["GEOID"].notna().sum()
unmatched = total - matched

total, matched, unmatched

In [ ]:
# 12a. Show Zillow city names that did not map to an ACS GEOID
sorted(
    zvhi_sd_la.loc[
        zvhi_sd_la["GEOID"].isna(),
        "City"
    ].unique()
)

In [ ]:
# 13. Remove ZIP rows that do not map to ACS places
    # dropped these 11 cities from the df

zvhi_aligned = zvhi_sd_la[zvhi_sd_la["GEOID"].notna()].copy()

zvhi_aligned.shape

In [ ]:
# 14. Raw ACS columns required to construct the 18 modeled variables

acs_vars = [

    # --------------------------------------------------
    # 1. Median household income
    # --------------------------------------------------
    "B19013_001E",  # Median household income (USD)

    # --------------------------------------------------
    # 2. Per capita income
    # --------------------------------------------------
    "B19301_001E",  # Per capita income (USD)

    # --------------------------------------------------
    # 3. Poverty rate (derived)
    # poverty_rate = below_poverty / poverty_universe
    # --------------------------------------------------
    "B17001_001E",  # Total population for whom poverty status is determined
    "B17001_002E",  # Population below poverty line

    # --------------------------------------------------
    # 4. Unemployment rate (derived)
    # unemployment_rate = unemployed / labor_force
    # --------------------------------------------------
    "B23025_003E",  # Civilian labor force
    "B23025_005E",  # Unemployed population

    # --------------------------------------------------
    # 5. Total population
    # --------------------------------------------------
    "B01003_001E",  # Total population

    # --------------------------------------------------
    # 7. Median age
    # --------------------------------------------------
    "B01002_001E",  # Median age (years)

    # --------------------------------------------------
    # 8. % Bachelor’s degree or higher (derived)
    # bachelors_or_higher = sum(bachelor+ degrees) / total 25+
    # --------------------------------------------------
    "B15003_001E",  # Total population age 25+
    "B15003_022E",  # Bachelor's degree
    "B15003_023E",  # Master's degree
    "B15003_024E",  # Professional degree
    "B15003_025E",  # Doctorate degree

    # --------------------------------------------------
    # 10. Total housing units
    # --------------------------------------------------
    "B25001_001E",  # Total housing units

    # --------------------------------------------------
    # 12. Vacancy rate (derived)
    # vacancy_rate = vacant_units / total_units
    # --------------------------------------------------
    "B25002_001E",  # Total housing units
    "B25002_003E",  # Vacant housing units

    # --------------------------------------------------
    # 13. Owner-occupied rate (derived)
    # owner_rate = owner_occupied / total_occupied
    # --------------------------------------------------
    "B25003_001E",  # Total occupied housing units
    "B25003_002E",  # Owner-occupied units

    # --------------------------------------------------
    # 14. % Built after 2000 (derived)
    # built_after_2000 = units_2000+ / total_units
    # --------------------------------------------------
    "B25034_001E",  # Total housing units (by year built)
    "B25034_010E",  # Built 2000–2009
    "B25034_011E",  # Built 2010 or later

    # --------------------------------------------------
    # 15. % Single-family detached (derived)
    # single_family_rate = detached_units / total_units
    # --------------------------------------------------
    "B25024_001E",  # Total housing units (by structure type)
    "B25024_002E",  # 1-unit detached structure

    # --------------------------------------------------
    # 16. Median home value
    # --------------------------------------------------
    "B25077_001E",  # Median home value (USD)

    # --------------------------------------------------
    # 17. Median gross rent
    # --------------------------------------------------
    "B25064_001E",  # Median gross rent (USD)

    # --------------------------------------------------
    # 18. % Owners cost-burdened (derived)
    # cost_burden_rate = owners paying >30% income on housing / total owners
    # --------------------------------------------------
    "B25091_001E",  # Total owner-occupied units with mortgage
    "B25091_010E",  # Owners paying 30–34.9%
    "B25091_011E",  # Owners paying 35–39.9%
    "B25091_012E"   # Owners paying 40%+
]

len(acs_vars)

In [ ]:
# 15. Pull ACS 5-year place-level data for 2009–2022,
#     automatically requesting only the variables that exist in each year.
#     Missing vars for early years will be created as NaN so columns align.

years = list(range(2009, 2023))  # 2009–2022

# ---------------------------------------------
# Helper 1: pick a working ACS5 endpoint for a year
# ---------------------------------------------
def get_acs5_base_url(year: int) -> str:
    candidates = [
        f"https://api.census.gov/data/{year}/acs/acs5",
        f"https://api.census.gov/data/{year}/acs5",
    ]
    for url in candidates:
        test = requests.get(url, params={"get": "NAME", "for": "place:*", "in": "state:06"})
        if test.status_code == 200:
            return url
    raise ValueError(f"No working ACS5 endpoint found for year {year}")

# ---------------------------------------------
# Helper 2: load that year's variable dictionary
# (this tells us what variables are valid for that year)
# ---------------------------------------------
def get_year_variable_set(year: int, base_url: str) -> set:
    vars_url = base_url + "/variables.json"
    r = requests.get(vars_url)
    if r.status_code != 200:
        raise ValueError(f"Could not fetch variables.json for {year} from {vars_url}")
    j = r.json()
    return set(j["variables"].keys())

acs_all_years = []

for year in years:
    print(f"Pulling {year}...")

    base_url = get_acs5_base_url(year)

    # Identify which of our desired raw vars actually exist for this year
    var_set = get_year_variable_set(year, base_url)
    available_vars = [v for v in acs_vars if v in var_set]
    missing_vars = [v for v in acs_vars if v not in var_set]

    # Show a short warning if anything is missing (expected in early years)
    if missing_vars:
        print(f"  {year}: missing {len(missing_vars)} vars (will be NaN). Example: {missing_vars[:3]}")

    # Request ONLY the vars that exist for this year
    params = {
        "get": ",".join(["NAME"] + available_vars),
        "for": "place:*",
        "in": "state:06",
        "key": API_KEY
    }

    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        print(f"  Error {response.status_code} for {year} using {base_url}")
        print(" ", response.text[:300])
        continue  # skip this year, try the next

    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # Convert pulled ACS numeric columns to numeric
    for col in available_vars:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Create missing columns (so every year has the same set of raw vars)
    for col in missing_vars:
        df[col] = pd.NA

    # Add year
    df["year"] = year

    # Reorder columns consistently
    df = df[["NAME"] + acs_vars + ["state", "place", "year"]]

    acs_all_years.append(df)

# Combine all years we successfully pulled
acs_raw_panel = pd.concat(acs_all_years, ignore_index=True)

acs_raw_panel.shape

In [ ]:
# 16. Construct the 18 modeled variables from the raw ACS panel
#     - Computes rates safely (avoids divide-by-zero)
#     - Renames columns to readable names
#     - Adds population_growth and housing_unit_growth using year-over-year change

import numpy as np

acs = acs_raw_panel.copy()

# -----------------------------
# Helper: safe divide (returns NaN if denom is 0/NaN)
# -----------------------------
def safe_div(numer, denom):
    numer = pd.to_numeric(numer, errors="coerce")
    denom = pd.to_numeric(denom, errors="coerce")
    return np.where((denom == 0) | (pd.isna(denom)), np.nan, numer / denom)

# -----------------------------
# 1–5, 7, 10, 16–17: direct pulls (rename)
# -----------------------------
acs["median_household_income"] = acs["B19013_001E"]   # 1
acs["per_capita_income"]       = acs["B19301_001E"]   # 2
acs["total_population"]        = acs["B01003_001E"]   # 5
acs["median_age"]              = acs["B01002_001E"]   # 7
acs["total_housing_units"]     = acs["B25001_001E"]   # 10
acs["median_home_value_acs"]   = acs["B25077_001E"]   # 16
acs["median_gross_rent"]       = acs["B25064_001E"]   # 17

# -----------------------------
# 3. Poverty rate
# -----------------------------
acs["poverty_rate"] = safe_div(acs["B17001_002E"], acs["B17001_001E"])  # 3

# -----------------------------
# 4. Unemployment rate
# -----------------------------
acs["unemployment_rate"] = safe_div(acs["B23025_005E"], acs["B23025_003E"])  # 4

# -----------------------------
# 8. % bachelor’s degree or higher (25+)
# -----------------------------
bachelors_or_higher_num = (
    pd.to_numeric(acs["B15003_022E"], errors="coerce") +  # bachelor's
    pd.to_numeric(acs["B15003_023E"], errors="coerce") +  # master's
    pd.to_numeric(acs["B15003_024E"], errors="coerce") +  # professional
    pd.to_numeric(acs["B15003_025E"], errors="coerce")    # doctorate
)
acs["pct_bachelors_or_higher"] = safe_div(bachelors_or_higher_num, acs["B15003_001E"])  # 8

# -----------------------------
# 9. % graduate degree (25+)
# -----------------------------
grad_num = (
    pd.to_numeric(acs["B15003_023E"], errors="coerce") +  # master's
    pd.to_numeric(acs["B15003_024E"], errors="coerce") +  # professional
    pd.to_numeric(acs["B15003_025E"], errors="coerce")    # doctorate
)
acs["pct_graduate_degree"] = safe_div(grad_num, acs["B15003_001E"])  # 9

# -----------------------------
# 12. Vacancy rate
# -----------------------------
acs["vacancy_rate"] = safe_div(acs["B25002_003E"], acs["B25002_001E"])  # 12

# -----------------------------
# 13. Owner-occupied rate (share of occupied units)
# -----------------------------
acs["owner_occupied_rate"] = safe_div(acs["B25003_002E"], acs["B25003_001E"])  # 13

# -----------------------------
# 14. % built after 2000
# -----------------------------
built_after_2000_num = (
    pd.to_numeric(acs["B25034_010E"], errors="coerce") +  # 2000-2009
    pd.to_numeric(acs["B25034_011E"], errors="coerce")    # 2010+
)
acs["pct_built_after_2000"] = safe_div(built_after_2000_num, acs["B25034_001E"])  # 14

# -----------------------------
# 15. % single-family detached
# -----------------------------
acs["pct_single_family_detached"] = safe_div(acs["B25024_002E"], acs["B25024_001E"])  # 15

# -----------------------------
# 18. % owners cost-burdened (>30%)
# NOTE: Using B25091 (owners w/ mortgage). Early years may be NaN (API missing vars).
# -----------------------------
cost_burden_num = (
    pd.to_numeric(acs["B25091_010E"], errors="coerce") +
    pd.to_numeric(acs["B25091_011E"], errors="coerce") +
    pd.to_numeric(acs["B25091_012E"], errors="coerce")
)
acs["pct_owners_cost_burdened"] = safe_div(cost_burden_num, acs["B25091_001E"])  # 18

# -----------------------------
# 6. Population growth (YoY, derived from panel)
# 11. Housing unit growth (YoY, derived from panel)
# Compute within each place across years
# -----------------------------
acs = acs.sort_values(["state", "place", "year"])

acs["population_growth"] = (
    acs.groupby(["state", "place"])["total_population"]
       .pct_change()
)

acs["housing_unit_growth"] = (
    acs.groupby(["state", "place"])["total_housing_units"]
       .pct_change()
)

# -----------------------------
# Keep only the final 18 variables + identifiers
# -----------------------------
acs_features = acs[[
    "NAME", "state", "place", "year",

    # 1–4
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",

    # 5–7
    "total_population",
    "population_growth",
    "median_age",

    # 8–9
    "pct_bachelors_or_higher",
    "pct_graduate_degree",

    # 10–15
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",

    # 16–18
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened"
]].copy()

acs_features.shape

In [ ]:
# 17. Keep only ACS rows for the places (cities) that appear in zvhi_aligned via GEOID

# GEOID in your workflow is the 7-digit place GEOID stored as a string
# ACS uses state + place; GEOID = state + place (for CA, state='06')

# Build the set of GEOIDs present in your Zillow-aligned dataset
geoids_used = set(zvhi_aligned["GEOID"].dropna().astype(str))

# Create GEOID in ACS features to filter with the same key
acs_features["GEOID"] = acs_features["state"].astype(str) + acs_features["place"].astype(str)

# Filter down to only the mapped cities
acs_features_sdla = acs_features[acs_features["GEOID"].isin(geoids_used)].copy()

# Sanity checks
print("Unique GEOIDs in Zillow:", len(geoids_used))
print("Unique GEOIDs in ACS after filter:", acs_features_sdla["GEOID"].nunique())
acs_features_sdla.shape

In [ ]:
# 18. Build monthly ZHVI panel + merge annual ACS features
#     - ZHVI stays monthly (no aggregation)
#     - We add a derived 'year' column from the monthly date
#     - Merge ACS on GEOID + year so each month inherits that year's ACS values

# ---------------------------------------------------------
# A) Identify ID columns and monthly ZHVI date columns
# ---------------------------------------------------------
id_cols = ["RegionID", "City", "CountyName", "State", "GEOID"]

# Date columns in your Zillow file are strings like "2018-01-31" (or similar)
date_cols = [c for c in zvhi_aligned.columns if isinstance(c, str) and c[:4].isdigit()]

print("ID cols found:", [c for c in id_cols if c in zvhi_aligned.columns])
print("Number of monthly ZHVI columns:", len(date_cols))
print("First 3 date cols:", date_cols[:3])
print("Last 3 date cols:", date_cols[-3:])

# ---------------------------------------------------------
# B) Convert ZHVI from wide format to long format (monthly rows)
#     wide: one row per city with many date columns
#     long: one row per city-month
# ---------------------------------------------------------
zvhi_long = zvhi_aligned[id_cols + date_cols].melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zhvi"
)

# ---------------------------------------------------------
# C) Parse the 'date' strings into real datetimes and create 'year'
#     - This does NOT change monthly ZHVI values
#     - It only adds a merge key for annual ACS data
# ---------------------------------------------------------
zvhi_long["date"] = pd.to_datetime(zvhi_long["date"], errors="coerce")
zvhi_long = zvhi_long.dropna(subset=["date"]).copy()

zvhi_long["year"] = zvhi_long["date"].dt.year

# ---------------------------------------------------------
# D) Ensure GEOID/year types match on both datasets
#     - Prevents silent merge failures due to dtype mismatch
# ---------------------------------------------------------
zvhi_long["GEOID"] = zvhi_long["GEOID"].astype(str)
zvhi_long["year"] = zvhi_long["year"].astype(int)

acs_features_sdla["GEOID"] = acs_features_sdla["GEOID"].astype(str)
acs_features_sdla["year"] = acs_features_sdla["year"].astype(int)

# ---------------------------------------------------------
# E) Merge annual ACS features onto monthly ZHVI rows
#     - Each month in a given year receives the same ACS values for that year
# ---------------------------------------------------------
zvhi_with_acs = zvhi_long.merge(
    acs_features_sdla.drop(columns=["NAME", "state", "place"]),
    on=["GEOID", "year"],
    how="left"
)

# ---------------------------------------------------------
# F) Sanity checks: confirm row counts and missingness
# ---------------------------------------------------------
print("ZHVI long shape:", zvhi_long.shape)
print("After merge shape:", zvhi_with_acs.shape)

# Pick one ACS feature as a match indicator
match_indicator = "median_household_income"
if match_indicator in zvhi_with_acs.columns:
    print("ACS missing rate (median_household_income):",
          zvhi_with_acs[match_indicator].isna().mean())
else:
    print(f"Warning: '{match_indicator}' not found in merged df")

zvhi_with_acs.head()

In [ ]:
# 18A. Inspect columns to identify the ZHVI monthly time-series column naming pattern

cols = zvhi_aligned.columns.astype(str).tolist()

print("Total columns:", len(cols))
print("First 30 columns:", cols[:30])
print("Last 30 columns:", cols[-30:])

# Show columns that contain a year-like pattern anywhere (e.g., '2018', '2020')
year_like = [c for c in cols if "20" in c]
print("Count of columns containing '20':", len(year_like))
print("Sample of columns containing '20':", year_like[:20])

In [ ]:
# 18B. Identify monthly ZHVI date columns in MM/DD/YYYY format

id_cols = ["RegionID", "City", "CountyName", "State", "GEOID"]

# Date columns look like '1/31/2000' so detect by the presence of '/'
date_cols = [c for c in zvhi_aligned.columns if isinstance(c, str) and "/" in c]

print("Monthly date columns:", len(date_cols))
print("First 3:", date_cols[:3])
print("Last 3:", date_cols[-3:])

In [ ]:
# 18C. Wide -> long (monthly), add year, merge ACS

# --- wide to long ---
zvhi_long = zvhi_aligned[id_cols + date_cols].melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zhvi"
)

# --- parse date + derive year (ZHVI remains monthly) ---
zvhi_long["date"] = pd.to_datetime(zvhi_long["date"], format="%m/%d/%Y", errors="coerce")
zvhi_long = zvhi_long.dropna(subset=["date"]).copy()
zvhi_long["year"] = zvhi_long["date"].dt.year.astype(int)

# --- type alignment for merge keys ---
zvhi_long["GEOID"] = zvhi_long["GEOID"].astype(str)
acs_features_sdla["GEOID"] = acs_features_sdla["GEOID"].astype(str)

# --- merge annual ACS onto monthly ZHVI using GEOID + year ---
zvhi_with_acs = zvhi_long.merge(
    acs_features_sdla.drop(columns=["NAME", "state", "place"]),
    on=["GEOID", "year"],
    how="left"
)

# --- sanity checks ---
print("ZHVI long shape:", zvhi_long.shape)
print("After merge shape:", zvhi_with_acs.shape)
print("ACS missing rate (median_household_income):", zvhi_with_acs["median_household_income"].isna().mean())

zvhi_with_acs.head()

In [ ]:
# 19. Keep only years >= 2009 (drop 2000–2008)
#     Do NOT forward-fill yet — we need real ACS values to compute growth correctly
    # 2009-2026 ZHVI data w 73,185 rows and 26 columns

zvhi_model_panel = zvhi_with_acs[
    zvhi_with_acs["year"] >= 2009
].copy()

print("Shape after dropping 2000–2008:", zvhi_model_panel.shape)

# Confirm ACS year coverage
print("Min year:", zvhi_model_panel["year"].min())
print("Max year:", zvhi_model_panel["year"].max())

# Check missingness before synthetic projection
print("Current missing rate (median_household_income):",
      zvhi_model_panel["median_household_income"].isna().mean())

In [ ]:
# 20. Compute 5-year average growth parameters per GEOID (2017–2022)
#     Use ONLY annual ACS (acs_features_sdla), not the monthly ZHVI panel.

# ------------------------------------------------------------
# A) Choose the window of real ACS years to estimate "typical" growth
# ------------------------------------------------------------
growth_start = 2017   # first year in the growth window
growth_end   = 2022   # last year in the growth window (last real ACS year we have)

In [ ]:
# 20b. Work from the annual ACS table (one row per GEOID-year)
# ------------------------------------------------------------
acs_annual = acs_features_sdla.copy()                     # make a copy so we don't modify original
acs_annual = acs_annual.sort_values(["GEOID", "year"])    # ensure rows are ordered for pct_change/diff

# Keep only the years we want for estimating growth
acs_growth_window = acs_annual[
    (acs_annual["year"] >= growth_start) &
    (acs_annual["year"] <= growth_end)
].copy()

# Quick sanity check: confirm the window is correct
print("Growth window:", acs_growth_window["year"].min(), "to", acs_growth_window["year"].max())
print("Unique GEOIDs in window:", acs_growth_window["GEOID"].nunique())

In [ ]:
# 20c. Define which variables are "levels" vs "rates"
#    Levels: project multiplicatively using % growth
#    Rates:  project additively using absolute change
# ------------------------------------------------------------
level_vars = [
    "median_household_income",   # 1
    "per_capita_income",         # 2
    "total_population",          # 5
    "median_age",                # 7
    "total_housing_units",       # 10
    "median_home_value_acs",     # 16
    "median_gross_rent",         # 17
]

rate_vars = [
    "poverty_rate",              # 3
    "unemployment_rate",         # 4
    "pct_bachelors_or_higher",   # 8
    "pct_graduate_degree",       # 9
    "vacancy_rate",              # 12
    "owner_occupied_rate",       # 13
    "pct_built_after_2000",      # 14
    "pct_single_family_detached",# 15
    "pct_owners_cost_burdened",  # 18
]

# (Optional) sanity check: confirm columns exist
missing = [c for c in level_vars + rate_vars if c not in acs_growth_window.columns]
if missing:
    raise ValueError(f"Missing expected ACS feature columns: {missing}")

In [ ]:
# 20d. (FIX). Average the yearly % changes to get one growth rate per GEOID

level_growth_avg = (
    level_growth
    # group by GEOID and average the YoY % changes across the growth window years
    .groupby(acs_growth_window["GEOID"])
    .mean(numeric_only=True)
    # convert GEOID from index back to a regular column
    .reset_index()
)

# Optional: make sure the first column is actually named 'GEOID'
# (it should be, but this guarantees it)
level_growth_avg = level_growth_avg.rename(columns={level_growth_avg.columns[0]: "GEOID"})

# Rename columns so it's obvious these are growth rates (g = growth)
level_growth_avg = level_growth_avg.rename(columns={c: f"{c}_g" for c in level_vars})

In [ ]:
# 20e. Compute yearly absolute changes for rate variables
#      Then average those changes across 2017–2022

# ---------------------------------------------
# Step 1: Compute year-over-year absolute change within each GEOID
# ---------------------------------------------
rate_change = (
    acs_growth_window
        .groupby("GEOID")[rate_vars]
        .diff()   # (current year - prior year)
)

# ---------------------------------------------
# Step 2: Average those yearly changes per GEOID
# ---------------------------------------------
rate_change_avg = (
    rate_change
        .groupby(acs_growth_window["GEOID"])
        .mean(numeric_only=True)
        .reset_index()
)

# Ensure first column is named GEOID
rate_change_avg = rate_change_avg.rename(
    columns={rate_change_avg.columns[0]: "GEOID"}
)

# Rename columns to indicate annual delta (d = delta)
rate_change_avg = rate_change_avg.rename(
    columns={c: f"{c}_d" for c in rate_vars}
)

rate_change_avg.head()

In [ ]:
# 20f. Recombine it all
acs_growth_params = level_growth_avg.merge(
    rate_change_avg,
    on="GEOID",
    how="outer"
)

acs_growth_params.shape

In [ ]:
# 21. Create synthetic ACS annual rows for 2023–2026

projection_years = [2023, 2024, 2025, 2026]

# ------------------------------------------
# A) Extract 2022 real ACS values (anchor)
# ------------------------------------------
acs_2022 = acs_features_sdla[
    acs_features_sdla["year"] == 2022
].copy()

print("2022 anchor rows:", acs_2022.shape)

In [ ]:
# 21b. Merge growth parameters onto 2022 base
# ------------------------------------------
acs_projection_base = acs_2022.merge(
    acs_growth_params,
    on="GEOID",
    how="left"
)

print("Projection base shape:", acs_projection_base.shape)

In [ ]:
# 21c. Generate synthetic rows year-by-year
# ------------------------------------------
synthetic_rows = []

for year in projection_years:
    
    years_ahead = year - 2022
    
    df_year = acs_projection_base.copy()
    df_year["year"] = year
    
    # ---- Level variables (compound growth) ----
    for col in level_vars:
        g_col = f"{col}_g"
        df_year[col] = df_year[col] * ((1 + df_year[g_col]) ** years_ahead)
    
    # ---- Rate variables (add annual delta) ----
    for col in rate_vars:
        d_col = f"{col}_d"
        df_year[col] = df_year[col] + (df_year[d_col] * years_ahead)
    
    synthetic_rows.append(
        df_year[acs_features_sdla.columns]  # keep only original ACS feature columns
    )

In [ ]:
# 21d. Combine synthetic years
# ------------------------------------------
acs_synthetic = pd.concat(synthetic_rows, ignore_index=True)

print("Synthetic ACS rows:", acs_synthetic.shape)
acs_synthetic.head()

In [ ]:
# 21e. Check 2023-2025 values are truly unique for a specified column

test_geoid = "0600212"   # replace with any GEOID you want to inspect

acs_synthetic[
    (acs_synthetic["GEOID"] == test_geoid) &
    (acs_synthetic["year"].isin([2023, 2024, 2025]))
][["GEOID", "year", "median_household_income"]].sort_values("year")how

In [ ]:
# 22. Combine real and synthetic ACS annual tables with source tagging

# ---------------------------------------
# A) Tag real ACS data
# ---------------------------------------
acs_real = acs_features_sdla.copy()
acs_real["acs_source"] = "real"

# ---------------------------------------
# B) Tag synthetic ACS data
# ---------------------------------------
acs_synthetic_tagged = acs_synthetic.copy()
acs_synthetic_tagged["acs_source"] = "synthetic"

# ---------------------------------------
# C) Combine into one annual ACS table
# ---------------------------------------
acs_full_annual = pd.concat(
    [acs_real, acs_synthetic_tagged],
    ignore_index=True
)

# Sanity check
print("Combined annual ACS shape:", acs_full_annual.shape)
print(acs_full_annual["acs_source"].value_counts())
acs_full_annual.head()

In [ ]:
# 22a. Merge combined ACS annual table (real + synthetic) onto monthly ZHVI panel
# ------------------------------------------------------------
# A) Ensure merge keys are the same type on both tables
# ------------------------------------------------------------
acs_full_annual["GEOID"] = acs_full_annual["GEOID"].astype(str)
acs_full_annual["year"]  = acs_full_annual["year"].astype(int)

zvhi_model_panel["GEOID"] = zvhi_model_panel["GEOID"].astype(str)
zvhi_model_panel["year"]  = zvhi_model_panel["year"].astype(int)

In [ ]:
# 22b. Drop existing ACS columns from zvhi_model_panel (prevents duplicates)
# ------------------------------------------------------------
acs_feature_cols = [
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",
    "total_population",
    "population_growth",
    "median_age",
    "pct_bachelors_or_higher",
    "pct_graduate_degree",
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened",
    "acs_source",
]

zvhi_model_panel_noacs = zvhi_model_panel.drop(
    columns=[c for c in acs_feature_cols if c in zvhi_model_panel.columns],
    errors="ignore"
)

In [ ]:
# 22c. Merge annual ACS onto monthly panel by GEOID + year
# ------------------------------------------------------------
zvhi_final_panel = zvhi_model_panel_noacs.merge(
    acs_full_annual,
    on=["GEOID", "year"],
    how="left"
)

In [ ]:
# 22d. Sanity checks: confirm synthetic years are present
# ------------------------------------------------------------
print("zvhi_final_panel shape:", zvhi_final_panel.shape)
print("acs_source counts (incl NaN):")
print(zvhi_final_panel["acs_source"].value_counts(dropna=False))

# Optional: check 2023–2025 specifically
print("\nacs_source counts for 2023–2025:")
print(zvhi_final_panel.loc[zvhi_final_panel["year"].between(2023, 2025), "acs_source"]
      .value_counts(dropna=False))

In [ ]:
# 22e. Diagnose unmatched rows (acs_source is NaN)

unmatched = zvhi_final_panel[zvhi_final_panel["acs_source"].isna()].copy()

print("Unmatched rows:", unmatched.shape[0])

print("\nUnmatched by year:")
print(unmatched["year"].value_counts().sort_index())

print("\nUnmatched by GEOID (top 20):")
print(unmatched["GEOID"].value_counts().head(20))

# show a few examples so we can see the pattern 
unmatched[["RegionID", "City", "CountyName", "State", "GEOID", "year", "date"]].head(10)

In [ ]:
# 23. Build 2018–2025 dataset

# Build clean 2018–2025 dataset (monthly, zero ACS missing values)

panel_2018_2025 = zvhi_final_panel[
    (zvhi_final_panel["year"] >= 2018) &
    (zvhi_final_panel["year"] <= 2025)
].copy()

print("2018–2025 shape:", panel_2018_2025.shape)

acs_feature_cols = [
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",
    "total_population",
    "population_growth",
    "median_age",
    "pct_bachelors_or_higher",
    "pct_graduate_degree",
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened"
]

total_nan = panel_2018_2025[acs_feature_cols].isna().sum().sum()

print("Total ACS NaNs (2018–2025):", total_nan)

In [ ]:
# 23b. How many unique zips in 2018-2025

print("Unique GEOIDs (2018–2025):", panel_2018_2025["GEOID"].nunique())

In [ ]:
# 24. what 2009 columns are na
# Filter 2009 rows only
panel_2009 = zvhi_final_panel[zvhi_final_panel["year"] == 2009].copy()

# Count NaNs per ACS feature column
na_by_column_2009 = (
    panel_2009[acs_feature_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
)

# Show only columns that actually have missing values
na_by_column_2009 = na_by_column_2009[na_by_column_2009 > 0]

na_by_column_2009

In [ ]:
# 24a. what 2010 columns are na
# Filter 2010 rows only
panel_2010 = zvhi_final_panel[zvhi_final_panel["year"] == 2010].copy()

print("Total rows in 2010:", len(panel_2010))

# Count NaNs per ACS feature column
na_by_column_2010 = (
    panel_2010[acs_feature_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
)

# Show only columns that actually have missing values
na_by_column_2010 = na_by_column_2010[na_by_column_2010 > 0]

na_by_column_2010

In [ ]:
# 25a. what columns are in panel_2018_2025
# Check if both ZHVI and ACS variables exist in 2018–2025 panel

print("Columns in panel_2018_2025:")
print(panel_2018_2025.columns.tolist())

In [ ]:
# 25b. confirm ZHVI is still monthly (it is)
print("Unique months in 2018–2025:",
      panel_2018_2025["date"].nunique())

print("Rows per year (should be 12 × ZIP count):")
print(panel_2018_2025.groupby("year").size())

In [ ]:
# 25c. Print first 20 rows of panel_2015_2018
# Display first 20 rows with all columns visible

# Temporarily show all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

panel_2018_2025.head(20)

In [ ]:
# 25d. panel_2015_2018 columns (no RegionName)
panel_2018_2025.columns

In [ ]:
# 25e. Check if RegionName column exists in the original ZHVI dataframe

print("RegionName exists in zhvi:", "RegionName" in zvhi.columns)

# If it exists, show first few values
if "RegionName" in zvhi.columns:
    print(zvhi["RegionName"].head())

In [ ]:
# 25f. Check if RegionName column exists in ZHVI_aligned
"RegionName" in zvhi_aligned.columns

In [ ]:
#25g. Check firtst 2 rows of ZHVI_aligned
zvhi_aligned[["RegionID", "RegionName", "City", "State"]].head(20)

In [ ]:
# 26. Build cleamn Region to Zip Identifier
# RegionID -> ZIP lookup (RegionName)
zip_lookup = (
    zvhi_aligned[["RegionID", "RegionName"]]
    .dropna(subset=["RegionID", "RegionName"])
    .drop_duplicates(subset=["RegionID"])
    .copy()
)

# Ensure ZIP is a 5-char string (keeps leading zeros)
zip_lookup["RegionName"] = zip_lookup["RegionName"].astype(str).str.zfill(5)

print("Unique RegionIDs in lookup:", zip_lookup["RegionID"].nunique())
zip_lookup.head()

In [ ]:
# 26a. Attach zip_lookup to panel_2018_2025:
# Add ZIP into panel_2018_2025 (if not already there)
panel_2018_2025 = panel_2018_2025.merge(zip_lookup, on="RegionID", how="left")

# Sanity check
print("ZIP missing rate:", panel_2018_2025["RegionName"].isna().mean())
panel_2018_2025[["RegionID", "RegionName", "City", "CountyName", "State"]].head(10)

In [ ]:
# 26b. Attach zip lookup to other df's
# Examples (replace with your actual dataframe names)

zvhi_long      = zvhi_long.merge(zip_lookup, on="RegionID", how="left")
zvhi_with_acs  = zvhi_with_acs.merge(zip_lookup, on="RegionID", how="left")
zvhi_model_panel = zvhi_model_panel.merge(zip_lookup, on="RegionID", how="left")

In [ ]:
# 26c. Sanity check. Print any unmatched zips (0)

missing_ids = panel_2018_2025.loc[panel_2018_2025["RegionName"].isna(), "RegionID"].unique()
missing_ids[:20], len(missing_ids)

In [ ]:
# 27. export to Excel for viewing purposes

panel_2018_2025.to_excel("panel_2018_2025.xlsx", index=False)

In [ ]:
import os
os.getcwd()

In [ ]:
import pandas as pd

#28. read in csv file as df

panel_2018_2025_v2 = pd.read_csv("panel_2018_2025(Sheet1).csv")

#28a. remove whitespaces before the column name
panel_2018_2025_v2.columns = panel_2018_2025_v2.columns.str.strip()
#print(panel_2018_2025_v2.columns)

#28b. write to csv to view data 
panel_2018_2025_v2.to_csv("panel_2018_2025_v2.csv",index = False)

In [12]:
#29. Re-Encode data to numerical/categorical types

# Encoding Numerical/Categorical features

#-------------------------------------------------------------------
# Features:
# ID: zips, geoid 
# categorical: city, county, acs_souce
# numerical: $ and % features 
#-------------------------------------------------------------------

#29a. create reference for id, categorical, date, $ col, and % cols

features = {
    "id_cols": ["RegionName", "GEOID"],
    "categorical_cols": ["City", "CountyName", "acs_source"],
    "datetime_cols": ["date"],
    "currency_cols": [
        "zhvi",
        "median_household_income",
        "per_capita_income",
        "median_home_value_acs",
        "median_gross_rent"
        
    ],
    "percent_cols": [
        "poverty_rate",
        "unemployment_rate",
        "population_growth",
        "pct_bachelors_or_higher",
        "pct_graduate_degree",
        "housing_unit_growth",
        "vacancy_rate",
        "owner_occupied_rate",
        "pct_built_after_2000",
        "pct_single_family_detached",
        "pct_owners_cost_burdened"
    ]
}

#29b. create function to strip the $ and convert object to numeric

def clean_currency_df(X):
    X = pd.DataFrame(X).copy()
    for col in X.columns:
        X[col] = pd.to_numeric(
            X[col].astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip(),
            errors="coerce"
        )
    return X
    
#29c. create function to strip the % and convert object to numeric

def clean_percent_df(X):
    X = pd.DataFrame(X).copy()
    for col in X.columns:
        X[col] = pd.to_numeric(
            X[col].astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip(),
            errors="coerce"
        )
    return X

#29d. apply to data frame

df = panel_2018_2025_v2.copy()

for col in features["currency_cols"]:
    df[col] = clean_currency_df(df[col])

for col in features["percent_cols"]:
    df[col] = clean_percent_df(df[col])

for col in features["datetime_cols"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in features["categorical_cols"]:
    df[col] = df[col].astype("category")

#29e. (optional) rename columns with $ and % in column name 

df = df.rename(columns={col: f"{col}_$" for col in features["currency_cols"]})
df = df.rename(columns={col: f"{col}_%" for col in features["percent_cols"]})

NameError: name 'panel_2018_2025_v2' is not defined

In [ ]:
#29f. create copy to view
panel_2018_2025_v3 = df.copy()
panel_2018_2025_v3 = panel_2018_2025_v3.sort_values(["RegionName", "date"])

In [ ]:
#view in csv
panel_2018_2025_v3.to_csv("panel_2018_2025_v3.csv",index = False)

In [ ]:
#30. Address missing values 

#----------------------------------------------------------------
# encoded as ($666,666,666 or extremely large values)
# - identify which values are NaN
# - identify which are coded as $666,666,666 or extremely large values
# - re-encode the incorrect values to NaN
#----------------------------------------------------------------
import numpy as np
#30a. Missing Values in median household income

missing_vals = panel_2018_2025_v3[
    panel_2018_2025_v3["median_household_income_$"].isna()
][["RegionName","City","CountyName","date","median_household_income_$"]]

missing_vals.to_csv("missing_val.csv",index = False)
print("number of missing values:" , missing_vals["median_household_income_$"].isna().count())


#30b. wrong encoded values in median household income ($666,666,666 or extremely large values E+11/12/20)

#look at range to see if income is reasonable
print("Range of Median Household Income: ", panel_2018_2025_v3["median_household_income_$"].describe())

# identify incorrect encoded median incomes
wrong_encoding = panel_2018_2025_v3[panel_2018_2025_v3["median_household_income_$"] > 500000][["RegionName","City","median_household_income_$"]]
print("Incorrect encoded values: ", wrong_encoding)

#number of incorrect encodings
print("Number of wrong encodings: ", wrong_encoding["RegionName"].count())

#30c. convert the wrong values to NaN
panel_2018_2025_v3.loc[panel_2018_2025_v3["median_household_income_$"] > 500000,"median_household_income_$"] = np.nan

#sanity check 
panel_2018_2025_v3["median_household_income_$"].isna().sum()

In [ ]:
#30d. write to csv file

panel_2018_2025_v3.to_csv("panel_2018_2025_v3.csv",index = False)

In [ ]:
#31. Imputing values for median_household_income_$

#------------------------------------------------------------------------------------------------------------------
#two cases for imputation: 
#      1. Boulevard: missing all years 
#          - impute using county-year median income
#      2. Lake Hughes and Jacumba have some data but missing data for multiple years
#          - impute using linear interpolation

#alternative: 240/34000 = 0.007 
#      1. very small proportion, can consider dropping not sure if imputation makes a difference at this small rate
#      2. Con:  we lose 3 cities pretty much

#--------------------------------------------------------------------------------------------------------------------


#31a. See how much data is missing over the 7 years for each zip code
panel_2018_2025_v3.loc[
    panel_2018_2025_v3["median_household_income_$"].isna(),
    "RegionName"
].unique()

zip_miss = panel_2018_2025_v3.groupby("RegionName")["median_household_income_$"].apply(lambda x: x.isna().sum())
zip_miss.to_csv("zip_miss.csv") 


#31b. Create imputed dataset

panel_2018_2025_v3 = panel_2018_2025_v3.copy().sort_values(["RegionName", "year", "date"])

# imputes for zipcodes of Lake Hughes and Jacumba using linear inp(91934 & 93532)

panel_2018_2025_v3["median_household_income_$"] = (panel_2018_2025_v3.groupby("RegionName")["median_household_income_$"]
    .transform(lambda x: x.interpolate(limit_direction="both"))
)

# imputes for Boulevard (91905)

county_year_income = (
    panel_2018_2025_v3
    .groupby(["CountyName", "year"])["median_household_income_$"]
    .median()
)

county_year_fill = (
    panel_2018_2025_v3[["CountyName", "year"]]
    .apply(tuple, axis=1)
    .map(county_year_income)
)

panel_2018_2025_v3["median_household_income_$"] = (
    panel_2018_2025_v3["median_household_income_$"].fillna(county_year_fill)
)

# view imputed dataset 

imputed_panelv3 = panel_2018_2025_v3.copy()

In [ ]:
#31c. view file
imputed_panelv3.to_csv("imputed_panelv3.csv",index = False)

#sanity check 
imputed_panelv3["median_household_income_$"].isna().sum()

In [ ]:
# 32. address missing values in zhvi 

#--------------------------------------------------------------------------------------------------------------------------
# Three ways to address:
# 
#  1. Use subsetted data from 2019-2025 since zipcode 920008 has a sample size of 80 while the rest have 96
#
#  2. Use subset of 2019-2025 for predicting, then after predictions go back and impute with interpolation to fill in missing 2018 year
#
#  3. We could drop zipcode 920008, but seems unnecessary since we have enough data to still forecast (2019-2025)

# Note: for single random missing values in zhvi we can drop those (sample sizes are still pretty close in size)
#----------------------------------------------------------------------------------------------------------------------------

# identify which zip codes have missing values in zhvi
zhvi_miss = imputed_panelv3.groupby("RegionName")["zhvi_$"].apply(lambda x: x.isna().sum())
zhvi_miss.to_csv("zhvi_miss.csv") 

#total number and proportion missing in zhvi (proportion is small we could drop it)
print("Missing values in imputed: ", zhvi_miss.sum()," ", panel_2018_2025_v3["zhvi_$"].isna().mean())

sample_size = imputed_panelv3.groupby("RegionName")["zhvi_$"].count()
sample_size.to_csv("sample_size.csv")

#---------------------------------------------------------------------
# 920008 has a sample size of 80 while the rest have 96
# which suggests we have a class imbalance
#---------------------------------------------------------------------

#32a. subset data to be 2019-2025 because otherwise we have a class imbalance 
panel_2019_2025 = imputed_panelv3[imputed_panelv3["year"] >= 2019].copy()

#check missing values in zhvi-$
print("Missing values in subset: ", panel_2019_2025["zhvi_$"].isna().sum()," ", panel_2019_2025["zhvi_$"].isna().mean())
zhvi_miss2019 = panel_2019_2025.groupby("RegionName")["zhvi_$"].apply(lambda x: x.isna().sum())
zhvi_miss2019.to_csv("zhvi_miss2019.csv") 

#32b. drop the missing values since sample sizes across zipcodes are similar and not substantially different (between 90-96)
panel_2019_2025 = panel_2019_2025.dropna(subset=["zhvi_$"])
print("NA after dropping: ", panel_2019_2025["zhvi_$"].isna().sum())

In [ ]:
# write subset to CSV to see 5 missing
panel_2019_2025.to_csv("panel_2019_2025.csv",index = False)

In [ ]:
#33. add time series feature: 

#------------------------------------------------------------
#  - Lag
#  - Rolling Growth rates
#  - Rolling means
#  - Rolling volatilities
#  - Price Acceleration
#  - Distance from Rolling Trend
#  - Price to Income Ratio
#  - rolling growth rate (12) vs population growth
#--------------------------------------------------------------------------

# Parse dates and sort (chronological order for lags)
panel_2019_2025["date"] = pd.to_datetime(panel_2019_2025["date"])
panel_2019_2025 = panel_2019_2025.sort_values(["RegionName", "date"])
#panel_2019_2025.head(10)

In [ ]:
#33a. Add lag features

panel_2019_2025["zhvi_lag1"] = panel_2019_2025.groupby("RegionName")["zhvi_$"].shift(1)
panel_2019_2025["zhvi_lag3"] = panel_2019_2025.groupby("RegionName")["zhvi_$"].shift(3)
panel_2019_2025["zhvi_lag6"] = panel_2019_2025.groupby("RegionName")["zhvi_$"].shift(6)
panel_2019_2025["zhvi_lag12"] = panel_2019_2025.groupby("RegionName")["zhvi_$"].shift(12)

#panel_2019_2025.head(10)

In [ ]:
#33b. add growth rate features
import numpy as np

# add rolling growth/depreciation rates for quarterly and annual
panel_2019_2025['growth_3mo'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].pct_change(3)
panel_2019_2025['growth_12mo'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].pct_change(12)

#panel_2019_2025.head(10)

In [ ]:
#33c. add Rolling means

panel_2019_2025['zhvi_roll3'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(3).mean())
panel_2019_2025['zhvi_roll6'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(6).mean())
panel_2019_2025['zhvi_roll12'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(12).mean())

#anel_2019_2025.head(10)

In [ ]:
#33d. add in Rolling volatilities

panel_2019_2025['zhvi_vol_3'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(3).std())
panel_2019_2025['zhvi_vol_6'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(6).std())
panel_2019_2025['zhvi_vol_12'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].transform(lambda x: x.rolling(12).std())

#panel_2019_2025.head(10)

In [ ]:
#33e.add in Price Acceleration (inflection points in the market switch from $+/$- or $-/$+)

panel_2019_2025['price_change'] = panel_2019_2025.groupby('RegionName')['zhvi_$'].diff()
panel_2019_2025['price_acceleration'] = panel_2019_2025.groupby('RegionName')['price_change'].diff()

#panel_2019_2025.head(10)

In [ ]:
#33f. add in Distance from Rolling Trend (the relative deviation from the mean trend) 

# 1    -> above long-term trend 
# 1 <  -> below trend 

#add in distance from mean trend
panel_2019_2025['price_vs_roll12'] = panel_2019_2025['zhvi_$'] /panel_2019_2025['zhvi_roll12']

#panel_2019_2025.head(10)

In [ ]:
#33g. Price to Income Ratio (housing affordability pressure)

#add in Price to Income Ratio
panel_2019_2025['price_income_ratio'] = panel_2019_2025['zhvi_$'] / panel_2019_2025['median_household_income_$']

#panel_2019_2025.head(10)

In [ ]:
#33h. add in rolling growth rate (12) vs population growth (supply and demand pressure)

# add in growth vs pop
panel_2019_2025['growth_vs_pop'] = panel_2019_2025['growth_12mo'] - panel_2019_2025['population_growth_%']

#panel_2019_2025.head(10)

In [ ]:
#33i. Write final dataset for analysis to csv 

#----------------------------------------------
# Final Dataset: panel_2019_2025
#----------------------------------------------

panel_2019_2025.to_csv("panel_2019_2025.csv",index = False)

In [ ]:
# 34. fit Baseline Models

#----------------------------------------------------------
# 
#   1. Random Walk with a Drift
#
#   2. ARIMA/ARMA 
#
#   3. Linear Mixed Models with Random Effects
#-----------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt


###############################################################

#34a. Model 1:  Random Walk 

################################################################

# subset final dataset 
rw_2019_2025 = panel_2019_2025[["RegionName", "date", "zhvi_$"]].copy()
rw_2019_2025 = rw_2019_2025.sort_values(["RegionName","date"])

#split the data in train and test 

split_date = "2024-01-01"

rw_train = rw_2019_2025[rw_2019_2025["date"] < split_date].copy()
rw_test  = rw_2019_2025[rw_2019_2025["date"] >= split_date].copy()

# model: RW for zipcodes

rw_forecasts = []

#loop through each zipcode
for zipcode, train_zip in rw_train.groupby("RegionName"):

    #get matching test set
    test_zip = rw_test[rw_test["RegionName"] == zipcode].copy()

    # compute the drift
    y_rwtrain = train_zip["zhvi_$"].values
    #avg_drift = (y_rwtrain[1:] - y_rwtrain[:1]).mean() (drift)
    last_rwtrain_val = y_rwtrain[-1]

    #forecast forward
    #h = range(1, len(test_zip) + 1)
    #preds = [last_rwtrain_val + avg_drift* step for step in h]  (for with a drift)
    preds = [last_rwtrain_val for _ in range(len(test_zip))]

    test_zip["rw_pred"] = preds

    #store predicted forecasts
    rw_forecasts.append(test_zip)

#combine into one data frame
rw_res = pd.concat(rw_forecasts, ignore_index = True)
rw_res.to_csv("predicted_forecasts.csv",index = False)

#plot one time series 

for zip_ts in [90002,92014,92010]:
    
    train_one = rw_train[rw_train["RegionName"] == zip_ts].copy()
    test_pred_one = rw_res[rw_res["RegionName"] == zip_ts].copy()

    plt.figure(figsize=(10, 4))
    
    # training actuals
    plt.plot(train_one["date"], train_one["zhvi_$"], label="Train Actual")
    
    # test actuals
    plt.plot(test_pred_one["date"], test_pred_one["zhvi_$"], label="Test Actual")
    
    # test predictions
    plt.plot(test_pred_one["date"], test_pred_one["rw_pred"], label="RW Forecast")
    
    plt.title(f"Random Walk  Forecast for ZIP {zip_ts}")
    plt.xlabel("Year")
    plt.ylabel("ZHVI")
    plt.legend()
    plt.show()

In [ ]:
###############################################################

#34b. Model 2:  ARIMA/ARMA 

################################################################

#34b_1. plot the series to see if it is covariance stationary 

#--------------------------------------------------------------------
# Plot to see if it is covariance stationary.
# - ideally there should be no upward trend if covariance stationary
# - if upward trend we violated: 
#        1. constant mean 
#        2. constant variance
#   
#    we can fix this by either log transforming, differencing, or both if needed
#
#--------------------------------------------------------------------

#34c_1. diagnostics:

#-------------------------------------------------------------------------------------------------------------------------------------------
# - ACF      ( ACF and PACF give us parameters and form of the ARIMA)
# - PACF 

# ADF p-value: > 0.05 not stationary so d = 1 (needs differencing)
#
# Based on the ACF and PACF plots:
#       - we have AR(1) characteristics since the ACF has a slow exponential decay and the PACF cuts off after lag 1
#       - No MA(q) characteristics
#       - planning to fit ARIMA (1,1,0)
#---------------------------------------------------------------------------------------------------------------------------------------------
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima

# split data
split_date = "2024-01-01"

arima_train = rw_2019_2025[rw_2019_2025["date"] < split_date].copy()
arima_test  = rw_2019_2025[rw_2019_2025["date"] >= split_date].copy()

# start with one zipcode first
zip_codes  = [90002,92014,92010]

for zip_code in zip_codes:
    # subset 
    train_arima = arima_train[arima_train["RegionName"] == zip_code].copy()
    test_arima  = arima_test[arima_test["RegionName"] == zip_code].copy()

    # sort by date
    train_arima = train_arima.sort_values("date")
    test_arima = test_arima.sort_values("date")

    # define series
    y_train_arima = train_arima["zhvi_$"]
    y_test_arima = test_arima["zhvi_$"]

    # skip if too few observations
    if len(y_train_arima.dropna()) < 10:
        print(f"Skipping ZIP {zip_code}: not enough observations")
        continue

    #print(f"\n===== ZIP {zip_code} =====")
    #print(train_arima.head())
    #print(train_arima.tail())

    # --------------------------
    # 1. plot time series
    # --------------------------
    plt.figure(figsize=(10, 4))
    plt.plot(train_arima["date"], y_train_arima)
    plt.title(f"Training ZHVI for ZIP {zip_code}")
    plt.xlabel("Year")
    plt.ylabel("ZHVI")
    plt.grid(True)
    plt.show()

    # --------------------------
    # 2. ADF test
    # --------------------------
    adf_result = adfuller(y_train_arima.dropna())
    print(f"ADF Statistic for ZIP {zip_code}: {adf_result[0]:.4f}")
    print(f"p-value for ZIP {zip_code}: {adf_result[1]:.4f}")

    # --------------------------
    # 3. ACF and PACF
    # --------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    plot_acf(y_train_arima.dropna(), lags=24, ax=axes[0])
    axes[0].set_title(f"ACF - ZIP {zip_code}")
    axes[0].set_xlabel("Lag")
    axes[0].set_ylabel("Correlation")

    plot_pacf(y_train_arima.dropna(), lags=24, ax=axes[1], method="ywm")
    axes[1].set_title(f"PACF - ZIP {zip_code}")
    axes[1].set_xlabel("Lag")
    axes[1].set_ylabel("Correlation")

    plt.tight_layout()
    plt.show()

In [ ]:
#34d_1. fit the ARIMA (1,1,0)

#---------------------------------------------------------------------------
# Based on the ACF and PACF of the zipcodes the time series need differencing
# hence we need d = 1, no MA portion is present but AR(1) is -> ARIMA(1,1,0)
#
# Diagnostics: 
#       - ACF      
#       - PACF 
#       - Ljung Boxes for autocorrelation 
#----------------------------------------------------------------------------

# split data
split_date = "2024-01-01"
arima_train = rw_2019_2025[rw_2019_2025["date"] < split_date].copy()
arima_test  = rw_2019_2025[rw_2019_2025["date"] >= split_date].copy()

# get all zipcodes
zip_codes = arima_train["RegionName"].dropna().unique()

results_summary = []
forecast_dict = {}

zip_codes  = [90002,92014,92010]

for zip_code in zip_codes:
    # subset one zipcode
    train_arima = arima_train[arima_train["RegionName"] == zip_code].copy()
    test_arima  = arima_test[arima_test["RegionName"] == zip_code].copy()

    # sort by date
    train_arima = train_arima.sort_values("date")
    test_arima  = test_arima.sort_values("date")

    # set date index (helpful for forecasting)
    train_arima = train_arima.set_index("date")
    test_arima = test_arima.set_index("date")

    # define series
    y_train_arima = train_arima["zhvi_$"].dropna()
    y_test_arima  = test_arima["zhvi_$"].dropna()

    try:
        # fit ARIMA(1,1,0), ARIMA(2,1,0), and ARIMA(1,1,1)
        model = ARIMA(y_train_arima, order=(1, 1, 0))
        model2 = ARIMA(y_train_arima, order=(2, 1, 0))
        model3 = ARIMA(y_train_arima, order=(1, 1, 1))
        
        fitted_model = model.fit()
        fitted_model2 = model2.fit()
        fitted_model3 = model3.fit()

        # forecast for length of test set
        n_future = len(test_arima)  #2024 - 2025

        #forecast for ARIMA(1,1,0)
        forecast_obj = fitted_model.get_forecast(steps=n_future)
        future_forecast = forecast_obj.predicted_mean
        future_ci = forecast_obj.conf_int()

        #forecast for ARIMA(2,1,0)
        forecast_obj2 = fitted_model2.get_forecast(steps=n_future)
        future_forecast2 = forecast_obj2.predicted_mean
        future_ci2 = forecast_obj2.conf_int()

        #forecast for ARIMA(1,1,1)
        forecast_obj3 = fitted_model3.get_forecast(steps=n_future)
        future_forecast3 = forecast_obj3.predicted_mean
        future_ci3 = forecast_obj3.conf_int()

        # residuals from fitted model

        # Residuals for ARIMA(1,1,0),ARIMA(2,1,0),ARIMA(1,1,1)
        residuals = fitted_model.resid.dropna()
        residuals2 = fitted_model2.resid.dropna()
        residuals3 = fitted_model3.resid.dropna()
        
         # plot forecast vs actual
        # plot all 3 models side by side
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

        # ---------------- ARIMA(1,1,0) ----------------
        axes[0].plot(y_train_arima.index, y_train_arima, label="Train")
        axes[0].plot(y_test_arima.index, y_test_arima, label="Test")
        axes[0].plot(future_forecast.index, future_forecast, label="ARIMA(1,1,0) Forecast", marker="+")
        axes[0].fill_between(future_ci.index,future_ci.iloc[:, 0],future_ci.iloc[:, 1],alpha=0.2,label="95% CI")
        axes[0].set_title(f"ZIP {zip_code}: ARIMA(1,1,0)")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("ZHVI")
        axes[0].grid(True)
        axes[0].legend()

        # ---------------- ARIMA(2,1,0) ----------------
        axes[1].plot(y_train_arima.index, y_train_arima, label="Train")
        axes[1].plot(y_test_arima.index, y_test_arima, label="Test")
        axes[1].plot(future_forecast2.index, future_forecast2, label="ARIMA(2,1,0) Forecast", marker="+")
        axes[1].fill_between(future_ci2.index,future_ci2.iloc[:, 0],future_ci2.iloc[:, 1],alpha=0.2,label="95% CI")
        axes[1].set_title(f"ZIP {zip_code}: ARIMA(2,1,0)")
        axes[1].set_xlabel("Year")
        axes[1].grid(True)
        axes[1].legend()

        # ---------------- ARIMA(1,1,1) ----------------
        axes[2].plot(y_train_arima.index, y_train_arima, label="Train")
        axes[2].plot(y_test_arima.index, y_test_arima, label="Test")
        axes[2].plot(future_forecast3.index, future_forecast3, label="ARIMA(1,1,1) Forecast", marker="+")
        axes[2].fill_between(future_ci3.index,future_ci3.iloc[:, 0],future_ci3.iloc[:, 1],alpha=0.2,label="95% CI")
        axes[2].set_title(f"ZIP {zip_code}: ARIMA(1,1,1)")
        axes[2].set_xlabel("Year")
        axes[2].grid(True)
        axes[2].legend()

        plt.tight_layout()
        plt.show()

        #------------ ACF and PACF of residuals--------------------
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        plot_acf(residuals, lags=24, ax=axes[0])
        axes[0].set_title(f"Residual ACF ARIMA(1,1,0) - ZIP {zip_code}")
        axes[0].set_xlabel("Lag")
        axes[0].set_ylabel("Correlation")

        plot_acf(residuals2, lags=24, ax=axes[1])
        axes[1].set_title(f"Residual ACF ARIMA(2,1,0)- ZIP {zip_code}")
        axes[1].set_xlabel("Lag")
        axes[1].set_ylabel("Correlation")

        plot_acf(residuals3, lags=24, ax=axes[2])
        axes[2].set_title(f"Residual ACF ARIMA(1,1,1)- ZIP {zip_code}")
        axes[2].set_xlabel("Lag")
        axes[2].set_ylabel("Correlation")

        plt.tight_layout()
        plt.show()
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        plot_pacf(residuals, lags=24, ax=axes[0], method="ywm")
        axes[0].set_title(f"Residual PACF ARIMA(1,1,0) - ZIP {zip_code}")
        axes[0].set_xlabel("Lag")
        axes[0].set_ylabel("Correlation")

        plot_pacf(residuals2, lags=24, ax=axes[1], method="ywm")
        axes[1].set_title(f"Residual PACF ARIMA(2,1,0)- ZIP {zip_code}")
        axes[1].set_xlabel("Lag")
        axes[1].set_ylabel("Correlation")

        plot_pacf(residuals3, lags=24, ax=axes[2], method="ywm")
        axes[2].set_title(f"Residual PACF ARIMA(1,1,1)- ZIP {zip_code}")
        axes[2].set_xlabel("Lag")
        axes[2].set_ylabel("Correlation")

        plt.tight_layout()
        plt.show()

        #----------- Ljung-Box test---------------------------------

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        lb_results = acorr_ljungbox(residuals.dropna(),lags=range(1, 25),return_df=True)
        lb_results2 = acorr_ljungbox(residuals2.dropna(),lags=range(1, 25),return_df=True)
        lb_results3 = acorr_ljungbox(residuals3.dropna(),lags=range(1, 25),return_df=True)
    
        axes[0].plot(lb_results.index, lb_results["lb_pvalue"], marker="o")
        axes[0].axhline(y=0.05, linestyle="--")
        axes[0].set_title(f"Ljung-Box p-values ARIMA(1,1,0) - ZIP {zip_code}")
        axes[0].set_xlabel("Lag")
        axes[0].set_ylabel("p-value")

        axes[1].plot(lb_results2.index, lb_results2["lb_pvalue"], marker="o")
        axes[1].axhline(y=0.05, linestyle="--")
        axes[1].set_title(f"Ljung-Box p-values ARIMA(2,1,0) - ZIP {zip_code}")
        axes[1].set_xlabel("Lag")
        axes[1].set_ylabel("p-value")

        axes[2].plot(lb_results3.index, lb_results3["lb_pvalue"], marker="o")
        axes[2].axhline(y=0.05, linestyle="--")
        axes[2].set_title(f"Ljung-Box p-values ARIMA(1,1,1) - ZIP {zip_code}")
        axes[2].set_xlabel("Lag")
        axes[2].set_ylabel("p-value")

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Could not fit ZIP {zip_code}: {e}")

#print(fitted_model.params) # AR coefficient ~ 0 pretty much a random walk since past changes don't help predict future changes by much.

In [ ]:
#####################################################################


#34c. Model 3: Linear Mixed Model 


#####################################################################

from datetime import datetime
import seaborn as sns

lmm_df = panel_2019_2025.copy()
lmm_df.head()

#34c_1. setup training and testing sets

#convert zipcodes as categories
lmm_df["RegionName"] = lmm_df["RegionName"].astype(str)

#sort and create time index
lmm_df = lmm_df.sort_values(["RegionName", "date"])
lmm_df["date"] = pd.to_datetime(lmm_df["date"])
lmm_df["time"] = lmm_df.groupby("RegionName").cumcount()
lmm_df.head()

#--------------------------------split into training and testing--------------------------------------- 
split_date = "2024-01-01"

lmm_train = lmm_df[lmm_df["date"] < split_date].copy()
lmm_test = lmm_df[lmm_df["date"] >= split_date].copy()

#outcome
y_lmm_train = lmm_train["zhvi_$"]
y_lmm_test  = lmm_test["zhvi_$"]

#create correlation heatmap to help identify correlated features to exclude

numeric_df = lmm_train.select_dtypes(include=["number"]).copy()

corr_matrix = numeric_df.corr()

plt.figure(figsize=(14,12))
sns.heatmap(
    corr_matrix,
    cmap="Spectral",
    annot = True,
    fmt=".1f",
    center=0,
    linewidths=0.5,
    square = True,
    cbar_kws = {"shrink": 0.8}
)

plt.title("Correlation Heatmap of Features (Training Data)")
plt.show()

In [ ]:
# see correlations with zhvi 
corr = numeric_df.corr()["zhvi_$"].drop("zhvi_$")
corr.sort_values(key=abs, ascending=False)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

#34b_1. Set up LMM Model 

# select initial features for model

features = [
    "time",                     #market direction over time
    "zhvi_lag1",                #momentum - captures how persistence market is
    "price_income_ratio",       #affordability - captures demand constraints
    "vacancy_rate_%",           #market structure - captures supply
    "pct_bachelors_or_higher_%" # demographics - captures long-term demand/quality
    
]


#double check VIF

# keep only training rows and selected columns
vif_df = lmm_train[features].dropna().copy()

# add intercept
X = sm.add_constant(vif_df)

# calculate VIF
vif_results = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

# print(vif_results)

#                      feature        VIF                                           feature        VIF
# 0                      const  20.841442                   # 0                      const  96.788434
# 1                       time   1.163735                   # 1                       time   1.244692
# 2                  zhvi_lag1   7.172826                   # 2                  zhvi_lag1  23.313351
# 3         price_income_ratio   4.646926                   # 3         price_income_ratio  17.404707
# 4             vacancy_rate_%   1.330526                   # 4  median_household_income_$   8.834531
# 5  pct_bachelors_or_higher_%   2.104812                   # 5  pct_bachelors_or_higher_%   2.522325
                                                            # 6             vacancy_rate_%   1.345565

#-------------------- Model Setup ------------------------------------------------------------------------

#model variables 
model_vars = [
    "RegionName",
    "zhvi_$",
    "time",
    "zhvi_lag1",
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]

#need to standardize the features because scales vary across the features
lmm_model_df = lmm_train[model_vars].dropna().copy()

lmm_model_df["vacancy_rate_%"] = lmm_model_df["vacancy_rate_%"] / 100
lmm_model_df["pct_bachelors_or_higher_%"] = lmm_model_df["pct_bachelors_or_higher_%"] / 100

scale_cols = [
    "zhvi_lag1",
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]

scaler = StandardScaler()
lmm_model_df[scale_cols] = scaler.fit_transform(lmm_model_df[scale_cols])


#------------------------------ Nested Models and Outputs -----------------------------------------------------

##############################################################################################
# Fitted Model 1: Initial Full Model
##############################################################################################

lmm_model = smf.mixedlm(
    "Q('zhvi_$') ~ time + zhvi_lag1 + price_income_ratio + Q('vacancy_rate_%') + Q('pct_bachelors_or_higher_%')",
    data=lmm_model_df,
    groups=lmm_model_df["RegionName"],
    re_formula= "~time"
)

result = lmm_model.fit(method="lbfgs")  
print("Initial Full Model: ", result.summary())

######################################################################################################
#Fitted Model 2 : Dropped Vacancy Rate because it was not significant
######################################################################################################

lmm_model2 = smf.mixedlm(
    "Q('zhvi_$') ~ time + zhvi_lag1 + price_income_ratio + Q('pct_bachelors_or_higher_%')",
    data=lmm_model_df,
    groups=lmm_model_df["RegionName"],
    re_formula= "~time"
)

result2 = lmm_model2.fit(method="lbfgs")  
print("Original Model Results: ", result2.summary())

###########################################################################################################
#   Fitted Model 3 : Log Transformed ZHVI  (Final Model used: Best one out of the 5)
############################################################################################################

lmm_model_df2 = lmm_model_df.copy()
lmm_model_df2 = lmm_train[model_vars].copy()

#log-transform both zhvi and zhvi_lag1 because they need to be on same scales and they need to be positive so select values > 0
lmm_model_df2["log_zhvi_$"] = np.where(
    lmm_model_df2["zhvi_$"] > 0,
    np.log(lmm_model_df2["zhvi_$"]),
    np.nan
)

lmm_model_df2["log_zhvi_lag1"] = np.where(
    lmm_model_df2["zhvi_lag1"] > 0,
    np.log(lmm_model_df2["zhvi_lag1"]),
    np.nan
)

lmm_model_df2["vacancy_rate_%"] = lmm_model_df2["vacancy_rate_%"] / 100
lmm_model_df2["pct_bachelors_or_higher_%"] = lmm_model_df2["pct_bachelors_or_higher_%"] / 100

scale_cols = [
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]

scaler = StandardScaler()
lmm_model_df2[scale_cols] = scaler.fit_transform(lmm_model_df2[scale_cols])

# drop any na values after log transformation
lmm_model_df2 = lmm_model_df2.replace([np.inf, -np.inf], np.nan).dropna()

#print(lmm_model_df2.shape)
#print(lmm_model_df2[[
#     "log_zhvi_$", "log_zhvi_lag1", "time",
#     "price_income_ratio", "pct_bachelors_or_higher_%"
# ]].isna().sum())

group_sizes = lmm_model_df2.groupby("RegionName").size()
lmm_model_df2 = lmm_model_df2[lmm_model_df2["RegionName"].isin(group_sizes[group_sizes >= 10].index)]

# print(group_sizes)
# print(lmm_model_df2)

#fit model 3
lmm_model3 = smf.mixedlm(
    "Q('log_zhvi_$') ~ time + log_zhvi_lag1 + price_income_ratio + Q('pct_bachelors_or_higher_%')",
    data=lmm_model_df2,
    groups=lmm_model_df2["RegionName"],
    re_formula= "~time"
)

result3 = lmm_model3.fit(method="lbfgs")  
print("Log Transformed Model Results: " ,result3.summary())

########################################################################################
#   Fitted Model 4 : added ahvi_lag2 to see if it improves ACF/PACF
#########################################################################################

#select features
lmm_model_df3 = lmm_train[[
    "RegionName",
    "date",
    "zhvi_$",
    "time",
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]].copy()

lmm_model_df3 = lmm_model_df3.sort_values(["RegionName", "date"])

#add lags to data frame
lmm_model_df3["zhvi_lag1"] = lmm_model_df3.groupby("RegionName")["zhvi_$"].shift(1)
lmm_model_df3["zhvi_lag2"] = lmm_model_df3.groupby("RegionName")["zhvi_$"].shift(2)

#log transform zhvi_$ and zhvi_lag1 and zhvi_lag2
lmm_model_df3["log_zhvi_$"] = np.log(lmm_model_df3["zhvi_$"])
lmm_model_df3["log_zhvi_lag1"] = np.log(lmm_model_df3["zhvi_lag1"])
lmm_model_df3["log_zhvi_lag2"] = np.log(lmm_model_df3["zhvi_lag2"])

# scale so converges and values arent inflated
lmm_model_df3["vacancy_rate_%"] = lmm_model_df3["vacancy_rate_%"] / 100
lmm_model_df3["pct_bachelors_or_higher_%"] = lmm_model_df3["pct_bachelors_or_higher_%"] / 100

scale_cols = [
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]

scaler = StandardScaler()
lmm_model_df3[scale_cols] = scaler.fit_transform(lmm_model_df3[scale_cols])

lmm_model_df3 = lmm_model_df3.replace([np.inf, -np.inf], np.nan).dropna()

# fit model 4
lmm_model4 = smf.mixedlm(
    "Q('log_zhvi_$') ~ time + log_zhvi_lag1 + log_zhvi_lag2 + price_income_ratio + Q('pct_bachelors_or_higher_%')",
    data=lmm_model_df3,
    groups=lmm_model_df3["RegionName"],
    re_formula="~time"
)

result4 = lmm_model4.fit(method="lbfgs")
print("Log Transformed with add zhvi_lag2 Results: ", result4.summary())

###############################################################################
# Fitted model 5 : log(price_income_ratio) for linearity issue
###############################################################################

#select features
lmm_model_df4 = lmm_train[[
    "RegionName",
    "date",
    "zhvi_$",
    "time",
    "price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]].copy()

lmm_model_df4 = lmm_model_df4.sort_values(["RegionName", "date"])

#add lags to data frame
lmm_model_df4["zhvi_lag1"] = lmm_model_df4.groupby("RegionName")["zhvi_$"].shift(1)


#log transform zhvi_$ and zhvi_lag1 and zhvi_lag2
lmm_model_df4["log_zhvi_$"] = np.log(lmm_model_df4["zhvi_$"])
lmm_model_df4["log_zhvi_lag1"] = np.log(lmm_model_df4["zhvi_lag1"])
lmm_model_df4["log_price_income_ratio"] = np.log(lmm_model_df4["price_income_ratio"])

# scale so converges and values arent inflated
lmm_model_df4["vacancy_rate_%"] = lmm_model_df4["vacancy_rate_%"] / 100
lmm_model_df4["pct_bachelors_or_higher_%"] = lmm_model_df4["pct_bachelors_or_higher_%"] / 100

scale_cols = [
    "log_price_income_ratio",
    "vacancy_rate_%",
    "pct_bachelors_or_higher_%"
]

scaler = StandardScaler()
lmm_model_df4[scale_cols] = scaler.fit_transform(lmm_model_df4[scale_cols])

lmm_model_df4 = lmm_model_df4.replace([np.inf, -np.inf], np.nan).dropna()

# fit model 5
lmm_model5 = smf.mixedlm(
    "Q('log_zhvi_$') ~ time + log_zhvi_lag1 + log_price_income_ratio + Q('pct_bachelors_or_higher_%')",
    data=lmm_model_df4,
    groups=lmm_model_df4["RegionName"],
    re_formula="~time"
)

result5 = lmm_model5.fit(method="lbfgs")
print("Log Transformed with add log(price_income_ratio) Results: ", result5.summary())

In [ ]:
# ---------------------------------Diagnostics---------------------------------------------------


# -----------------------------------------------
# Assumptions: 
# 1. linearity between predictor and response      - good
# 2. Normality of errors                           - okay, outliers might be pulling away from line, not severely violating
# 3. Normality of random effects                   - violated 
# 4. Homoscedasticity (constant variance)          - violated 
# 5. No multicollinearity                          - good
# 6. Independent observations                      - mostly white noise, some spikes around 10-15 that are above confidence band (not severely violating)
# -----------------------------------------------


# Residual vs Fitted: Linearity/Homoscedasticity

#scatterplots for linearity
lmm_model_df = lmm_model_df.copy()

predictors = [
    "time",
    "log_zhvi_lag1",
    "price_income_ratio",
    "pct_bachelors_or_higher_%"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, var in zip(axes, predictors):
    ax.scatter(lmm_model_df2[var], lmm_model_df2["log_zhvi_$"], alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel("log(ZHVI)")
    ax.set_title(f"log(ZHVI) vs {var}")

plt.tight_layout()
plt.show()

zips = ["90002", "92014", "92010"]


for region in zips:
    subset = lmm_model_df2[lmm_model_df2["RegionName"] == region]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for ax, var in zip(axes, predictors):
        ax.scatter(subset[var], subset["log_zhvi_$"], alpha=0.5)
        ax.set_title(f"{region}: log(ZHVI) vs {var}")
        ax.set_xlabel(var)
        ax.set_ylabel("log(ZHVI)")

    plt.tight_layout()
    plt.show()

#get residuals and fitted values

# -----model 2 ------------
lmm_model_df["fitted"] = result2.fittedvalues
lmm_model_df["resid"] = lmm_model_df["zhvi_$"] - lmm_model_df["fitted"]

# ---- model 3 -------------
lmm_model_df2["fitted"] = result3.fittedvalues
lmm_model_df2["resid"] = lmm_model_df2["log_zhvi_$"] - lmm_model_df2["fitted"]

fig, axes = plt.subplots(1, 2, figsize=(12,5))

# Residuals vs Fitted plot
axes[0].scatter(lmm_model_df["fitted"], lmm_model_df["resid"], alpha=0.3)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted for ZHVI Model")

axes[1].scatter(lmm_model_df2["fitted"], lmm_model_df2["resid"], alpha=0.3)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Fitted values")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residuals vs Fitted for log_ZHVI Model")

#########
# NOTE  : there is a funnel shape which suggests heterscedacity (violation) so we likely need to transform the response (log transform) 
#######


# Normality of errors and random effects 

#--------------- model 2 -------------------------
resid = lmm_model_df["resid"].dropna().values
# standardized residuals because scaling is off in qq-plot for errors
resid_std = (resid - resid.mean()) / resid.std()


# --------------- model 3 --------------------
resid2 = lmm_model_df2["resid"].dropna().values
# standardized residuals because scaling is off in qq-plot for errors
resid_std2 = (resid2 - resid2.mean()) / resid2.std()

fig, axes = plt.subplots(1, 2, figsize=(12,5))

# QQ Plot - heavy tails and skewed
sm.qqplot(resid_std, line="45", ax=axes[0])
axes[0].set_title("QQ Plot of Residuals for for ZHVI Model")

sm.qqplot(resid_std2, line="45", ax=axes[1])
axes[1].set_title("QQ Plot of Residuals for for log_ZHVI Model")

plt.tight_layout()
plt.show()

# random effects

# -------------- model 2 --------------------------------
re_df = pd.DataFrame(result2.random_effects).T

#rename columns (optional)
re_df.columns = ["Intercept_RE", "time_RE"]   # only if there are exactly 2 cols

intercept_re = re_df["Intercept_RE"]
time_re = re_df["time_RE"]

intercept_re_std = (intercept_re - intercept_re.mean()) / intercept_re.std()
time_re_std = (time_re - time_re.mean()) / time_re.std()


# -------------- model 3 --------------------------------
re_df2 = pd.DataFrame(result3.random_effects).T

#rename columns (optional)
re_df2.columns = ["Intercept_RE", "time_RE"]   # only if there are exactly 2 cols

intercept_re2 = re_df2["Intercept_RE"]
time_re2 = re_df2["time_RE"]

intercept_re_std2 = (intercept_re2 - intercept_re2.mean()) / intercept_re2.std()
time_re_std2 = (time_re2 - time_re2.mean()) / time_re2.std()

fig, axes = plt.subplots(1, 2, figsize=(12,5))

sm.qqplot(intercept_re_std, line="45", ax=axes[0])
axes[0].set_title("Random Intercepts for ZHVI Model")

sm.qqplot(time_re_std2, line="45", ax=axes[1])
axes[1].set_title("Random Slopes for log_ZHVI Model")

plt.tight_layout()
plt.show()


# autocorrelation in residuals - giving strong AR(1)
regions = ["90002","92014","92010"]


fig, axes = plt.subplots(len(regions), 2, figsize=(12, 4*len(regions)))

for i, region in enumerate(regions):
    
    subset = lmm_model_df[lmm_model_df["RegionName"] == region]
    resid = subset["resid"]

    plot_acf(resid, lags=20, ax=axes[i, 0])
    axes[i, 0].set_title(f"ACF - {region} for ZHVI Model")

    plot_pacf(resid, lags=20, ax=axes[i, 1])
    axes[i, 1].set_title(f"PACF - {region} for ZHVI Model")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(len(regions), 2, figsize=(12, 4*len(regions)))

for i, region in enumerate(regions):
    
    subset2 = lmm_model_df2[lmm_model_df2["RegionName"] == region]
    resid2 = subset2["resid"]

    plot_acf(resid2, lags=20, ax=axes[i, 0])
    axes[i, 0].set_title(f"ACF - {region} for log_ZHVI Model")

    plot_pacf(resid2, lags=20, ax=axes[i, 1])
    axes[i, 1].set_title(f"PACF - {region} for log_ZHVI Model")

plt.tight_layout()
plt.show()

# VIF for multicollinearity after fitting initial model 

#------------------- model 2 ----------------------------------------
lmm_vars = [
    "time",
    "zhvi_lag1",
    "price_income_ratio",
    "pct_bachelors_or_higher_%"
]


vif_lmm = lmm_train[lmm_vars].dropna().copy()
X = sm.add_constant(vif_lmm)

# calculate VIF
vif_lmm_res = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

#------------------- model 3 ----------------------------------------
lmm_vars2 = [
    "time",
    "log_zhvi_lag1",
    "price_income_ratio",
    "pct_bachelors_or_higher_%"
]


vif_lmm2 = lmm_model_df2[lmm_vars2].dropna().copy()
X2 = sm.add_constant(vif_lmm2)

# calculate VIF
vif_lmm_res2 = pd.DataFrame({
    "feature": X2.columns,
    "VIF": [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]
})

print(vif_lmm_res)
print(vif_lmm_res2)

In [ ]:
# 35. Unsupervised Models (PCA, K-means, t-sne)

#-----------------------------------------------------------------
#
# Principal Component Analysis (PCA)
#
#-----------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#############################################################################################
# 35a. PCA (Structural: how do regions differ in socioeconomic and housing characteristics? affordability)
#          ( Dynamic: do those same regions behave differently over time?)
#############################################################################################

df = panel_2019_2025.copy()

# ensure the data is sorted in chronological order by zipcode
df = df.sort_values(["RegionName","date"])

#create time component
df["time"] = df.groupby("RegionName").cumcount()

#------------------ split to train and testing sets -----------------------------------

split_date = "2024-01-01"

train_df = df[df["date"] < split_date].copy()
test_df  = df[df["date"] >= split_date].copy()

#------------------ structural and dynamic PCA Dataset ----------------------------------------------
# structural 
struct_features = [
    "median_household_income_$",
    "per_capita_income_$",
    "poverty_rate_%",
    "unemployment_rate_%",
    "median_age",
    "pct_bachelors_or_higher_%",
    "pct_graduate_degree_%",
    "total_population",
    "population_growth_%",
    "vacancy_rate_%",
    "owner_occupied_rate_%",
    "pct_built_after_2000_%",
    "median_home_value_acs_$",
    "median_gross_rent_$",
    "price_income_ratio"
]

df_struct = train_df.copy()

# convert % → proportions
pct_cols = [
    "poverty_rate_%",
    "unemployment_rate_%",
    "pct_bachelors_or_higher_%",
    "pct_graduate_degree_%",
    "vacancy_rate_%",
    "owner_occupied_rate_%"
]

for col in pct_cols:
    df_struct[col] = df_struct[col] / 100

# aggregate to one row per region
df_struct_region = df_struct.groupby("RegionName")[struct_features].mean().dropna()


# dynamic features 
dyn_features = [
    "growth_3mo",
    "growth_12mo",
    "zhvi_vol_3",
    "zhvi_vol_6",
    "zhvi_vol_12",
    "price_change",
    "price_acceleration",
    "price_vs_roll12",
    "growth_vs_pop"
]

df_dyn = train_df.copy()

df_dyn_region = df_dyn.groupby("RegionName")[dyn_features].mean().dropna()

#---------------------------- PCA ------------------------------------------------
scaler_struct = StandardScaler()
X_struct = scaler_struct.fit_transform(df_struct_region)

pca_struct = PCA(n_components=2)
X_struct_pca = pca_struct.fit_transform(X_struct)



scaler_dyn = StandardScaler()
X_dyn = scaler_dyn.fit_transform(df_dyn_region)

pca_dyn = PCA(n_components=2)
X_dyn_pca = pca_dyn.fit_transform(X_dyn)


#------------------------ Plots -------------------------------------------------

pc1_var = pca_struct.explained_variance_ratio_[0]
pc2_var = pca_struct.explained_variance_ratio_[1]

zips = [90002, 92014, 92010]

plt.figure(figsize=(6,5))
plt.scatter(X_struct_pca[:,0], 
            X_struct_pca[:,1], 
            c = df_struct_region["price_income_ratio"],
            cmap = "twilight",
            alpha=0.6)

for i, region in enumerate(df_struct_region.index):
    if region in zips:
        plt.text(
            X_struct_pca[i,0],
            X_struct_pca[i,1],
            region,
            fontsize=10,
            color="black"
        )

plt.title("Structural PCA (Train)")
plt.xlabel(f"PC1 ({pc1_var:.2%} variance)")
plt.ylabel(f"PC2 ({pc2_var:.2%} variance)")
plt.show()

pc1_var = pca_dyn.explained_variance_ratio_[0]
pc2_var = pca_dyn.explained_variance_ratio_[1]

plt.figure(figsize=(6,5))
plt.scatter(X_dyn_pca[:,0], 
            X_dyn_pca[:,1],
            c = df_dyn_region["growth_12mo"],
            cmap = "twilight",
            alpha=0.6
           )
for i, region in enumerate(df_dyn_region.index):
    if region in zips:
        plt.text(
            X_dyn_pca[i,0],
            X_dyn_pca[i,1],
            region,
            fontsize=10,
            color="black"
        )
            
plt.title("Dynamic PCA (Train)")
plt.xlabel(f"PC1 ({pc1_var:.2%} variance)")
plt.ylabel(f"PC2 ({pc2_var:.2%} variance)")
plt.show()

In [11]:
#############################################################################################
# 35b. K-Means ( Which regions belong in similar clusters?)
#############################################################################################

from sklearn.cluster import KMeans

X_struct_km = X_struct.copy()
X_dyn_km = X_dyn.copy()

#-------------- K - Means ---------------------------

# structural
kmeans = KMeans(n_clusters=3, random_state=42)
clusters_struct = kmeans.fit_predict(X_struct_km)

df_struct_region["cluster"] = clusters_struct

#dynamic
kmeans = KMeans(n_clusters=3, random_state=42)
clusters_dyn = kmeans.fit_predict(X_dyn_km)

df_dyn_region["cluster"] = clusters_dyn

# structural alignment
df_struct_region = df_struct_region.loc[df_struct_region.index]
# dynamic alignment
df_dyn_region = df_dyn_region.loc[df_dyn_region.index]

#-------------- Plot ------------------------------
zips = [90002, 92014, 92010]

plt.scatter(
    X_struct_pca[:,0],
    X_struct_pca[:,1],
    c=clusters_struct,
    cmap="Accent",
    alpha=0.7
)

for i, region in enumerate(df_struct_region.index):
    if region in zips:
        plt.text(
            X_struct_pca[i, 0],
            X_struct_pca[i, 1],
            region,
            fontsize=10
        )

plt.title("Structural Clusters (K-means)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

plt.scatter(
    X_dyn_pca[:,0],
    X_dyn_pca[:,1],
    c=clusters_dyn,
    cmap="Accent",
    alpha=0.7
)

for i, region in enumerate(df_dyn_region.index):
    if region in zips:
        plt.text(
            X_dyn_pca[i, 0],
            X_dyn_pca[i, 1],
            region,
            fontsize=10
        )

plt.title("Dynamic Clusters (K-means)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

NameError: name 'X_struct' is not defined

In [9]:
# 36. Statistical Model Evaluations

from sklearn.metrics import mean_squared_error, mean_absolute_error

def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calc_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


############################################################
    
#  36a.               Random Walk 
    
############################################################

rw_rmse = calc_rmse(rw_res["zhvi_$"], rw_res["rw_pred"])
rw_mae  = calc_mae(rw_res["zhvi_$"], rw_res["rw_pred"])

#print("Random Walk Overall Performance")
#print("RMSE:", rw_rmse)
#print("MAE :", rw_mae)

# RMSE, MAE by zipcodes
rw_zip_metrics = []

for zip_code, df_zip in rw_res.groupby("RegionName"):
    rw_zip_metrics.append({
        "RegionName": zip_code,
        "Model": "Random Walk",
        "RMSE": calc_rmse(df_zip["zhvi_$"], df_zip["rw_pred"]),
        "MAE": calc_mae(df_zip["zhvi_$"], df_zip["rw_pred"])
    })

rw_zip_metrics_df = pd.DataFrame(rw_zip_metrics)
#print(rw_zip_metrics_df.head())

rw_overall_metrics_df = pd.DataFrame([{
    "Model": "Random Walk",
    "RMSE": calc_rmse(rw_res["zhvi_$"], rw_res["rw_pred"]),
    "MAE": calc_mae(rw_res["zhvi_$"], rw_res["rw_pred"])
}])


#############################################################

# 36b.                       ARIMA

#############################################################

arima_results = []
arima_metrics = []

overall_arima_metrics = []

for model_col, model_name in [
    ("ARIMA_110_pred", "ARIMA(1,1,0)"),
    ("ARIMA_210_pred", "ARIMA(2,1,0)"),
    ("ARIMA_111_pred", "ARIMA(1,1,1)")
]:
    overall_arima_metrics.append({
        "Model": model_name,
        "RMSE": calc_rmse(arima_results_df["actual"], arima_results_df[model_col]),
        "MAE": calc_mae(arima_results_df["actual"], arima_results_df[model_col])
    })

overall_arima_metrics_df = pd.DataFrame(overall_arima_metrics)
#print(overall_arima_metrics_df)

# by zipcodes
zip_codes = [90002, 92014, 92010]

for zip_code in zip_codes:
    train_arima = arima_train[arima_train["RegionName"] == zip_code].copy()
    test_arima  = arima_test[arima_test["RegionName"] == zip_code].copy()

    train_arima = train_arima.sort_values("date").set_index("date")
    test_arima  = test_arima.sort_values("date").set_index("date")

    y_train_arima = train_arima["zhvi_$"].dropna()
    y_test_arima  = test_arima["zhvi_$"].dropna()

    try:
        model1 = ARIMA(y_train_arima, order=(1, 1, 0)).fit()
        model2 = ARIMA(y_train_arima, order=(2, 1, 0)).fit()
        model3 = ARIMA(y_train_arima, order=(1, 1, 1)).fit()

        n_future = len(y_test_arima)

        fc1 = model1.get_forecast(steps=n_future).predicted_mean
        fc2 = model2.get_forecast(steps=n_future).predicted_mean
        fc3 = model3.get_forecast(steps=n_future).predicted_mean

        # store row-level predictions
        temp = pd.DataFrame({
            "RegionName": zip_code,
            "date": y_test_arima.index,
            "actual": y_test_arima.values,
            "ARIMA_110_pred": fc1.values,
            "ARIMA_210_pred": fc2.values,
            "ARIMA_111_pred": fc3.values
        })

        arima_results.append(temp)

        # store metrics for this ZIP
        arima_metrics.append({
            "RegionName": zip_code,
            "Model": "ARIMA(1,1,0)",
            "RMSE": calc_rmse(y_test_arima, fc1),
            "MAE": calc_mae(y_test_arima, fc1),
            "AIC": model1.aic
        })

        arima_metrics.append({
            "RegionName": zip_code,
            "Model": "ARIMA(2,1,0)",
            "RMSE": calc_rmse(y_test_arima, fc2),
            "MAE": calc_mae(y_test_arima, fc2),
            "AIC": model2.aic
        })

        arima_metrics.append({
            "RegionName": zip_code,
            "Model": "ARIMA(1,1,1)",
            "RMSE": calc_rmse(y_test_arima, fc3),
            "MAE": calc_mae(y_test_arima, fc3),
            "AIC": model3.aic
        })

    except Exception as e:
        print(f"Could not fit ZIP {zip_code}: {e}")

arima_results_df = pd.concat(arima_results, ignore_index=True)
arima_metrics_df = pd.DataFrame(arima_metrics)

#print(arima_results_df.head())
#print(arima_metrics_df)

####################################################################################

#  36c.                     linear mixed model 

##################################################################################

panel_2019_2025 = panel_2019_2025.sort_values(["RegionName", "date"]).copy()

panel_2019_2025["date"] = pd.to_datetime(panel_2019_2025["date"])

# create time index per ZIP
panel_2019_2025["time"] = panel_2019_2025.groupby("RegionName").cumcount() + 1

split_date = pd.to_datetime("2024-01-01")

lmm_train = panel_2019_2025[pd.to_datetime(panel_2019_2025["date"]) < split_date].copy()
lmm_test  = panel_2019_2025[pd.to_datetime(panel_2019_2025["date"]) >= split_date].copy()

# keep only variables needed for Model 3
model3_vars = [
    "RegionName",
    "time",
    "date",
    "zhvi_$",
    "zhvi_lag1",
    "price_income_ratio",
    "pct_bachelors_or_higher_%",
    
]

train_m3 = lmm_train[model3_vars].copy()
train_m3["date"] = pd.to_datetime(train_m3["date"])

# log transforms
train_m3["log_zhvi_$"] = np.where(train_m3["zhvi_$"] > 0, np.log(train_m3["zhvi_$"]), np.nan)
train_m3["log_zhvi_lag1"] = np.where(train_m3["zhvi_lag1"] > 0, np.log(train_m3["zhvi_lag1"]), np.nan)

# convert percentages to proportions
train_m3["pct_bachelors_or_higher_%"] = train_m3["pct_bachelors_or_higher_%"] / 100

# scale predictors
scale_cols_m3 = ["price_income_ratio", "pct_bachelors_or_higher_%"]

scaler_m3 = StandardScaler()
train_m3[scale_cols_m3] = scaler_m3.fit_transform(train_m3[scale_cols_m3])

# clean data
train_m3 = train_m3.replace([np.inf, -np.inf], np.nan).dropna()

# optional: keep only ZIPs with enough observations
group_sizes = train_m3.groupby("RegionName").size()
valid_zips_m3 = group_sizes[group_sizes >= 10].index
train_m3 = train_m3[train_m3["RegionName"].isin(valid_zips_m3)].copy()

train_m3 = train_m3.loc[:, ~train_m3.columns.duplicated()]

# fit model 3
lmm_model3 = smf.mixedlm(
    "Q('log_zhvi_$') ~ time + log_zhvi_lag1 + price_income_ratio + Q('pct_bachelors_or_higher_%')",
    data=train_m3,
    groups=train_m3["RegionName"],
    re_formula="~time"
)

result3 = lmm_model3.fit(method="lbfgs")
#print(result3.summary())

# re-do log_zhvi_lag1 recursively

test_m3 = lmm_test[[
    "RegionName",
    "date",
    "zhvi_$",
    "time",
    "zhvi_lag1",
    "price_income_ratio",
    "pct_bachelors_or_higher_%"
]].copy()

test_m3 = test_m3.loc[:, ~test_m3.columns.duplicated()]
test_m3["date"] = pd.to_datetime(test_m3["date"])

# keep only ZIPs that were in training
test_m3 = test_m3[test_m3["RegionName"].isin(valid_zips_m3)].copy()

# convert percentages to proportions now
test_m3["pct_bachelors_or_higher_%"] = test_m3["pct_bachelors_or_higher_%"] / 100

test_m3 = test_m3.sort_values(["RegionName", "date"]).copy()\

#forecasts
lmm3_forecasts = []

for zip_code in test_m3["RegionName"].unique():
    
    test_zip = test_m3[test_m3["RegionName"] == zip_code].sort_values("date").copy()

    if len(test_zip) == 0:
        continue

    # use actual lagged value from the test set, not recursive predictions
    test_zip = test_zip[test_zip["zhvi_lag1"] > 0].copy()
    if len(test_zip) == 0:
        continue

    test_zip["log_zhvi_lag1"] = np.log(test_zip["zhvi_lag1"])

    # scale the predictors the same way as training
    exog_raw = test_zip[[
        "price_income_ratio",
        "pct_bachelors_or_higher_%"
    ]].copy()

    exog_scaled = scaler_m3.transform(exog_raw)

    test_zip["price_income_ratio"] = exog_scaled[:, 0]
    test_zip["pct_bachelors_or_higher_%"] = exog_scaled[:, 1]

    # build prediction frame
    pred_row = test_zip[[
        "RegionName",
        "time",
        "log_zhvi_lag1",
        "price_income_ratio",
        "pct_bachelors_or_higher_%"
    ]].copy()

    # predict in log scale
    log_preds = result3.predict(pred_row)

    # back-transform to dollar scale
    zhvi_preds = np.exp(log_preds)

    temp = pd.DataFrame({
        "RegionName": test_zip["RegionName"].values,
        "date": test_zip["date"].values,
        "actual": test_zip["zhvi_$"].values,
        "LMM3_pred": zhvi_preds.values
    })

    lmm3_forecasts.append(temp)

lmm3_results_df = pd.concat(lmm3_forecasts, ignore_index=True)
#print(lmm3_results_df.head())

#overall metrics

lmm3_overall_metrics_df = pd.DataFrame([{
    "Model": "LMM Model 3",
    "RMSE": calc_rmse(lmm3_results_df["actual"], lmm3_results_df["LMM3_pred"]),
    "MAE": calc_mae(lmm3_results_df["actual"], lmm3_results_df["LMM3_pred"])
}])

#print(lmm3_overall_metrics_df)

# by zip code
lmm3_zip_metrics = []

for zip_code, df_zip in lmm3_results_df.groupby("RegionName"):
    lmm3_zip_metrics.append({
        "RegionName": zip_code,
        "Model": "LMM Model 3",
        "RMSE": calc_rmse(df_zip["actual"], df_zip["LMM3_pred"]),
        "MAE": calc_mae(df_zip["actual"], df_zip["LMM3_pred"])
    })

lmm3_zip_metrics_df = pd.DataFrame(lmm3_zip_metrics)
#print(lmm3_zip_metrics_df.head())

NameError: name 'rw_res' is not defined

In [ ]:
#################################################################

# 36d.               FINAL COMPARISON TABLES

#################################################################

# overall 
model_comparison_df = pd.concat(
    [
        rw_overall_metrics_df,
        overall_arima_metrics_df,
        lmm3_overall_metrics_df
    ],
    ignore_index=True
).sort_values("RMSE")

print(model_comparison_df)

# by zipcode

#random walk
rw_zip_metrics_df = rw_zip_metrics_df[["RegionName", "Model", "RMSE", "MAE"]]
#arima
arima_zip_metrics_df = arima_metrics_df[["RegionName", "Model", "RMSE", "MAE"]]

# combine to make table
zip_model_comparison_df = pd.concat(
    [
        rw_zip_metrics_df,
        arima_zip_metrics_df,
        lmm3_zip_metrics_df
    ],
    ignore_index=True
).sort_values(["RegionName", "RMSE","MAE"])

#print(zip_model_comparison_df)


In [ ]:
# 37. Forecast Plots of top 3 models

# Random Walk
rw_plot_df = rw_res[["RegionName", "date", "zhvi_$", "rw_pred"]].copy()
rw_plot_df = rw_plot_df.rename(columns={
    "zhvi_$": "actual",
    "rw_pred": "pred"
})
rw_plot_df["Model"] = "Random Walk"

# ARIMA(2,1,0)
arima210_plot_df = arima_results_df[["RegionName", "date", "actual", "ARIMA_210_pred"]].copy()
arima210_plot_df = arima210_plot_df.rename(columns={
    "ARIMA_210_pred": "pred"
})
arima210_plot_df["Model"] = "ARIMA(2,1,0)"

# LMM Model 3
lmm3_plot_df = lmm3_results_df[["RegionName", "date", "actual", "LMM3_pred"]].copy()
lmm3_plot_df = lmm3_plot_df.rename(columns={
    "LMM3_pred": "pred"
})
lmm3_plot_df["Model"] = "LMM Model 3"

# Combine all top 3 models
top3_plot_df = pd.concat(
    [rw_plot_df, arima210_plot_df, lmm3_plot_df],
    ignore_index=True
)

top3_plot_df["date"] = pd.to_datetime(top3_plot_df["date"])

zip_list = [90002, 92014, 92010]
split_date = pd.to_datetime("2024-01-01")

 # by zipcodes
zip_list = [90002, 92014, 92010]

for zip_code in zip_list:
    plt.figure(figsize=(10, 4))

    actual_zip = panel_2019_2025[panel_2019_2025["RegionName"] == zip_code].copy()
    actual_zip["date"] = pd.to_datetime(actual_zip["date"])
    actual_zip = actual_zip.sort_values("date")
    actual_zip = actual_zip[actual_zip["date"] >= split_date]

    plt.plot(actual_zip["date"], actual_zip["zhvi_$"], label="Test Actual", linewidth=2)

    pred_zip = top3_plot_df[top3_plot_df["RegionName"] == zip_code].copy()

    for model_name in ["Random Walk", "ARIMA(2,1,0)", "LMM Model 3"]:
        model_df = pred_zip[pred_zip["Model"] == model_name].sort_values("date")
        plt.plot(model_df["date"], model_df["pred"], label=model_name, marker="o")

    plt.title(f"Top 3 Model Forecasts for ZIP {zip_code}")
    plt.xlabel("Date")
    plt.ylabel("ZHVI")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [10]:
# ----------------------------------------------------------
#  SUMMARY (FOR ML TEAM)
# ----------------------------------------------------------
# 1. Baseline models:
#    - Random Walk
#    - ARIMA(1,1,0), (2,1,0), (1,1,1)
#
# 2. Best-performing models:
#    - Random Walk (lowest RMSE)
#    - LMM Model 3 (lowest MAE)
#
# 3. Use these as benchmarks when evaluating ML models
#
# 4. Key insight:
#    - Housing prices are highly persistent
#    - Any ML model must outperform Random Walk to be useful
#
# 5. Prediction outputs:
#    - rw_res
#    - arima_results_df
#    - lmm3_results_df
# ----------------------------------------------------------